# AMST+ViT+HOG: A Hybrid Shape Descriptor for Classification and Retrieval

**Environment notes.** This notebook is designed to run end-to-end on a CPU-only environment; every model call uses a `DEVICE` variable that automatically selects CUDA when available and falls back to CPU otherwise, so no cell requires a GPU. On CPU-only hardware, the dominant cost is the repeated ViT-B/16 forward passes used for feature extraction and for the perturbation-based robustness and invariance evaluations (Cells 14-15, 27-28, 30-31); the remaining cells (feature engineering, SVM training, statistical tests, and plotting) are fast in comparison. As a rough planning guide on typical CPU hardware, the single-pass feature extraction cells (14-15, 30) each take on the order of minutes, while the repeated-perturbation cells (27, 28, 31) are the longest individual cells, each on the order of tens of minutes, since they re-embed several hundred to a few thousand corrupted images through ViT-B/16 across multiple noise/occlusion/transformation levels and repeats. Running the full notebook once end-to-end is expected to take a few hours in total; intermediate results are cached to disk (`*.npz` / `*_features.npz` files) so the notebook can be re-run without repeating completed feature-extraction steps. If a faster run is needed (e.g., for a quick sanity check before a full run), `N_ROBUST_REPEATS`, `N_ROBUST_TEST` (Cells 27-28) and `N_INV_REPEATS`, `N_INV_PER_LEVEL` (Cell 31) can be reduced; this trades off the tightness of the reported error bars for speed and should be reverted to the full settings for the results reported in the paper.


## Cell 1 — Install Dependencies


In [1]:
import subprocess, sys, importlib, warnings, os
warnings.filterwarnings('ignore')

def ensure(pkg, imp=None):
    nm = imp or pkg.replace('-','_').replace('PyWavelets','pywt').replace('scikit_learn','sklearn').replace('scikit_image','skimage').replace('opencv_python_headless','cv2').replace('pillow','PIL')
    try: importlib.import_module(nm)
    except:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])

for pkg in ['PyWavelets','ripser','scikit-image','scikit-learn','matplotlib','seaborn','scipy','numpy','pandas','tqdm','opencv-python-headless','pillow','torch','torchvision','timm','scikit-posthocs']:
    ensure(pkg)

from ripser import ripser as _rp
print('All dependencies ready.')


Installing PyWavelets...
Installing ripser...
Installing opencv-python-headless...
Installing timm...
Installing scikit-posthocs...
All dependencies ready.


## Cell 2 — All Imports & Reproducibility


In [2]:
import os, sys, re, copy, json, glob, warnings, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from collections import Counter, defaultdict

import cv2
from PIL import Image
import pywt
import scipy, scipy.stats, scipy.special
from scipy import ndimage
from scipy.interpolate import interp1d
from scipy.spatial import ConvexHull
from ripser import ripser
from persim import plot_diagrams

from skimage import transform
from skimage.filters import threshold_otsu
from skimage.morphology import closing, opening, disk, remove_small_objects, skeletonize
from skimage.measure import find_contours
from skimage.feature import hog, local_binary_pattern

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import SelectKBest, f_classif
import sklearn
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, average_precision_score)
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
from torchvision import transforms, models
import timm

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'PyTorch {torch.__version__}, timm {timm.__version__}')
print(f'SKLearn { sklearn.__version__}, NumPy {np.__version__}')
print(f'Seed: {SEED}')


PyTorch 2.9.0+cpu, timm 1.0.27
SKLearn 1.6.1, NumPy 2.0.2
Seed: 42


## Cell 3 — Load MPEG-7 CE-Shape-1 Part B + Kimia216


In [3]:
IMG_SIZE = (128, 128)
N_CONTOUR_PTS = 200

DATA_DIR = Path('mpeg7_data')
DATA_DIR.mkdir(exist_ok=True)
zip_cands = ['CV_Project_Data/MPEG7-Original.zip', 'MPEG7_CE-Shape-1_Part_B.zip']

valid_pat = re.compile(r'^(.+)-(\d+)$')
valid_gifs = [f for f in Path('MPEG7_CE-Shape-1_Part_B').rglob('*.gif') if valid_pat.match(f.stem)]
if len(valid_gifs) < 1400:
    zf = next((z for z in zip_cands if Path(z).exists()), None)
    if zf:
        print(f'Extracting {zf}...')
        import zipfile
        with zipfile.ZipFile(zf) as z:
            z.extractall(DATA_DIR)
        valid_gifs = [f for f in DATA_DIR.rglob('*.gif') if valid_pat.match(f.stem)]
    else:
        for d in ['.', 'MPEG7_CE-Shape-1_Part_B', 'mpeg7_data']:
            valid_gifs = [f for f in Path(d).rglob('*.gif') if valid_pat.match(f.stem)]
            if len(valid_gifs) >= 1400: break

valid_gifs = sorted(valid_gifs, key=lambda x: x.name)
labels_raw = [valid_pat.match(f.stem).group(1) for f in valid_gifs]
# Lowercase for cross-dataset consistency with Kimia labels
labels_lower = [l.lower() for l in labels_raw]
cc = Counter(labels_lower)
print(f'MPEG-7: {len(valid_gifs)} images, {len(cc)} classes')
if len(valid_gifs) < 1400:
    print(f'WARNING: Found {len(valid_gifs)} GIFs, expected 1400. Continuing with available data.')

le = LabelEncoder()
y_mpeg = le.fit_transform(labels_lower)

# Kimia216 data loading
KIMIA_ROOT_DIR = Path('kimia_data')
kimia_zip_cands = ['Kimia216-Original.zip', 'CV_Project_Data/Kimia216-Original.zip']

# Check for and extract Kimia216 zip file
kim_zf = next((z for z in kimia_zip_cands if Path(z).exists()), None)

# Check if Kimia data (any image file) already exists in KIMIA_ROOT_DIR or its subfolders
existing_kimia_files = list(KIMIA_ROOT_DIR.rglob('*.jpg')) + list(KIMIA_ROOT_DIR.rglob('*.png'))

if kim_zf and not existing_kimia_files:
    print(f'Extracting {kim_zf} to {KIMIA_ROOT_DIR}...')
    KIMIA_ROOT_DIR.mkdir(parents=True, exist_ok=True)
    import zipfile
    with zipfile.ZipFile(kim_zf) as z:
        z.extractall(KIMIA_ROOT_DIR)
    print(f"Extraction complete to {KIMIA_ROOT_DIR}.")
elif not kim_zf:
    print("Kimia216-Original.zip not found. Please upload it to the runtime.")
else:
    print(f"Kimia216 data already found in {KIMIA_ROOT_DIR} (or subdirectories).")

# Now, find all image files within KIMIA_ROOT_DIR (or its subfolders if the zip was nested)
kim_files = sorted(KIMIA_ROOT_DIR.rglob('*.jpg')) + sorted(KIMIA_ROOT_DIR.rglob('*.png'))

# If after extraction and initial search, no files are found, give a warning.
if not kim_files:
    print("WARNING: No Kimia216 image files (.jpg or .png) found within the expected directories.")

kim_labels = []
for f in kim_files:
    stem = f.stem
    m = re.match(r'^([a-zA-Z]+)', stem)
    kim_labels.append(m.group(1).lower() if m else stem)
le_kimia = LabelEncoder()
if len(kim_labels) > 0:
    y_kimia = le_kimia.fit_transform(kim_labels)
else:
    y_kimia = np.array([])

print(f'Kimia216: {len(kim_files)} images, {len(set(kim_labels))} classes')

Extracting MPEG7_CE-Shape-1_Part_B.zip...
MPEG-7: 1400 images, 70 classes
Extracting Kimia216-Original.zip to kimia_data...
Extraction complete to kimia_data.
Kimia216: 216 images, 18 classes


## Cell 4 — Image Preprocessing & Contour Extraction


In [4]:
PREPROC_CACHE = 'mpeg7_preprocessed_v2.npz'
PREPROC_CACHE_FALLBACK = Path('Updated/mpeg7_preprocessed_v2.npz')

def load_binarize(path):
    img = Image.open(str(path)).convert('L')
    img = img.resize(IMG_SIZE, Image.LANCZOS)
    arr = np.array(img, dtype=np.float32) / 255.0
    try: th = threshold_otsu(arr)
    except: th = 0.5
    bw = (arr < th).astype(bool)
    if bw.sum() < 0.02 * IMG_SIZE[0] * IMG_SIZE[1]:
        bw = ~bw
    bw = closing(bw, disk(2))
    bw = opening(bw, disk(1))
    bw = remove_small_objects(bw, min_size=50)
    return bw.astype(np.uint8)

def get_contour(bw, n_pts=200):
    cnts = find_contours(bw.astype(float), 0.5)
    if not cnts: return np.zeros((n_pts, 2))
    c = max(cnts, key=len)
    d = np.diff(c, axis=0)
    arc = np.r_[0, np.cumsum(np.hypot(d[:,0], d[:,1]))]
    if arc[-1] < 1e-8: return np.zeros((n_pts, 2))
    u = np.linspace(0, arc[-1], n_pts, endpoint=False)
    pts = np.column_stack([np.interp(u, arc, c[:,0]), np.interp(u, arc, c[:,1])])
    pts -= pts.mean(axis=0)
    rmax = np.sqrt((pts**2).sum(axis=1)).max()
    return pts / (rmax + 1e-10)

cache_found = None
if Path(PREPROC_CACHE).exists():
    cache_found = PREPROC_CACHE
elif PREPROC_CACHE_FALLBACK.exists():
    cache_found = str(PREPROC_CACHE_FALLBACK)
if cache_found:
    print(f'Loading preprocessed cache from {cache_found}...')
    cache = np.load(cache_found)
    all_images = cache['images']
    all_contours = cache['contours']
    print(f'Loaded {len(all_images)} images')
else:
    Path('MPEG7_CE-Shape-1_Part_B').mkdir(parents=True, exist_ok=True)
    print(f'Preprocessing {len(valid_gifs)} images...')
    all_images, all_contours = [], []
    for p in tqdm(valid_gifs, desc='Preprocess'):
        bw = load_binarize(p)
        c = get_contour(bw)
        all_images.append(bw)
        all_contours.append(c)
    all_images = np.array(all_images, dtype=np.uint8)
    all_contours = np.array(all_contours, dtype=np.float32)
    np.savez_compressed(PREPROC_CACHE, images=all_images, contours=all_contours)

print(f'Images: {all_images.shape}, Contours: {all_contours.shape}')


Preprocessing 1400 images...


Preprocess: 100%|██████████| 1400/1400 [00:06<00:00, 222.28it/s]


Images: (1400, 128, 128), Contours: (1400, 200, 2)


## Cell 5 — Kimia216 Preprocessing & Contour Extraction


In [5]:
KIMIA_PREPROC_CACHE = 'kimia216_preprocessed.npz'
if Path(KIMIA_PREPROC_CACHE).exists():
    kc = np.load(KIMIA_PREPROC_CACHE)
    kim_images = kc['images']
    kim_contours = kc['contours']
    print(f'Loaded Kimia216: {len(kim_images)} images')
else:
    kim_images, kim_contours = [], []
    for p in tqdm(kim_files, desc='Kimia Preprocess'):
        bw = load_binarize(p)
        c = get_contour(bw)
        kim_images.append(bw)
        kim_contours.append(c)
    kim_images = np.array(kim_images, dtype=np.uint8)
    kim_contours = np.array(kim_contours, dtype=np.float32)
    np.savez_compressed(KIMIA_PREPROC_CACHE, images=kim_images, contours=kim_contours)
print(f'Kimia Images: {kim_images.shape}, Contours: {kim_contours.shape}')


Kimia Preprocess: 100%|██████████| 216/216 [00:00<00:00, 298.67it/s]

Kimia Images: (216, 128, 128), Contours: (216, 200, 2)


## Cell 6 — Baseline Shape Descriptors (HOG, Zernike, Fourier, Wavelet, CSS, Shape Context)


In [6]:
def hog_descriptor(bw):
    bw96 = transform.resize(bw.astype(float), (96,96), anti_aliasing=True) > 0.5
    return hog(bw96.astype(np.float32), orientations=9,
               pixels_per_cell=(16,16), cells_per_block=(1,1), feature_vector=True)

def zernike_descriptor(bw, max_order=10):
    h,w = bw.shape
    yg,xg = np.mgrid[-1:1:1j*h, -1:1:1j*w]
    rho = np.sqrt(xg**2 + yg**2)
    theta = np.arctan2(yg, xg)
    mask = (rho <= 1.0) & (bw > 0)
    moments = []
    for n in range(max_order+1):
        for m in range(-n, n+1, 2):
            if (n-abs(m))%2 != 0: continue
            R = np.zeros_like(rho)
            for s in range((n-abs(m))//2+1):
                coef = ((-1)**s * scipy.special.factorial(n-s)) / (
                    scipy.special.factorial(s) *
                    scipy.special.factorial((n+abs(m))//2 - s) *
                    scipy.special.factorial((n-abs(m))//2 - s) + 1e-300)
                R += coef * rho**(n-2*s)
            V = R * np.exp(-1j * m * theta)
            moments.append(np.abs(np.sum(V[mask]*bw[mask])*(n+1)/np.pi))
    return np.array(moments[:36])

def fourier_descriptor(cnt, n_coeff=32):
    r = np.sqrt((cnt**2).sum(axis=1))
    F = np.fft.fft(r)
    mag = np.abs(F)
    denom = mag[1] if mag[1] > 1e-8 else mag.max()+1e-12
    mag_n = mag / denom
    return np.concatenate([mag_n[1:n_coeff+1][:-1], np.angle(F)[1:9]])

def wavelet_descriptor(cnt, wavelet='db4', level=4):
    r = np.sqrt((cnt**2).sum(axis=1))
    r = r - r.mean()
    max_lvl = pywt.dwt_max_level(len(r), wavelet)
    L = min(level, max_lvl)
    coeffs = pywt.wavedec(r, wavelet, level=L, mode='periodization')
    energies = np.array([np.sum(c**2) for c in coeffs])
    ev = energies / (energies.sum()+1e-12)
    out = np.zeros(5)
    out[:min(len(ev),5)] = ev[:5]
    return out

def css_descriptor(cnt, sigmas=[1,2,4,8,16,32]):
    x, yc = cnt[:,1], cnt[:,0]
    feats = []
    for sigma in sigmas:
        xs = ndimage.gaussian_filter1d(x, sigma, mode='wrap')
        ys = ndimage.gaussian_filter1d(yc, sigma, mode='wrap')
        x1=np.gradient(xs); x2=np.gradient(x1)
        y1=np.gradient(ys); y2=np.gradient(y1)
        k = (x1*y2-x2*y1)/(x1**2+y1**2+1e-12)**1.5
        feats += [float(np.sum(np.diff(np.sign(k))!=0)), float(np.mean(np.abs(k)))]
    return np.array(feats)

def shape_context(cnt, n_r=5, n_theta=12):
    N = len(cnt)
    step = max(1, N//64)
    pts = cnt[::step]
    n = len(pts)
    dx = pts[:,1:2]-pts[np.newaxis,:,1]
    dy = pts[:,0:1]-pts[np.newaxis,:,0]
    dist = np.sqrt(dx**2+dy**2+1e-12)
    angles = np.arctan2(dy,dx)
    log_dist = np.log(dist/(dist.max()+1e-12)+1e-12)
    r_bins = np.linspace(log_dist.min()-0.01, 0.01, n_r+1)
    t_bins = np.linspace(-np.pi, np.pi, n_theta+1)
    H = np.zeros(n_r*n_theta)
    for i in range(n):
        mi = np.arange(n)!=i
        h,_,_ = np.histogram2d(log_dist[i,mi], angles[i,mi], bins=[r_bins,t_bins])
        H += h.flatten()
    return H / (H.sum()+1e-12)

t_bw, t_cnt = all_images[0], all_contours[0]
print(f'Baseline dims: HOG={len(hog_descriptor(t_bw))}, Zernike={len(zernike_descriptor(t_bw))}, Fourier={len(fourier_descriptor(t_cnt))}, Wavelet={len(wavelet_descriptor(t_cnt))}, CSS={len(css_descriptor(t_cnt))}, SC={len(shape_context(t_cnt))}')


Baseline dims: HOG=324, Zernike=36, Fourier=39, Wavelet=5, CSS=12, SC=60


## Cell 7 — AMST C1: APCFW+ (160-d) Rotation-Invariant Radial Fourier-Wavelet


In [7]:
def c1_apcfw_plus(cnt, K=60, n_wb=40):
    r = np.sqrt((cnt**2).sum(axis=1))
    x, yc = cnt[:,1], cnt[:,0]
    Fr = np.fft.fft(r)
    mag_r = np.abs(Fr)
    denom = mag_r[1] if mag_r[1] > 1e-8 else mag_r.max() + 1e-12
    fd_r = mag_r[1:K+1] / denom
    n_star = int(np.argmax(mag_r[1:K+1]))+1
    rho = n_star / K
    wv = 'db6' if rho<0.10 else ('db4' if rho<0.20 else ('db2' if rho<0.35 else 'haar'))
    x1=np.gradient(x); y1=np.gradient(yc)
    x2=np.gradient(x1); y2=np.gradient(y1)
    kappa = (x1*y2-x2*y1)/(x1**2+y1**2+1e-12)**1.5
    kc = kappa - kappa.mean()
    max_lvl = pywt.dwt_max_level(len(kc), wv)
    L = max(1, min(7, max_lvl))
    coeffs = pywt.wavedec(kc, wv, level=L, mode='periodization')
    energies = np.array([np.sum(c**2) for c in coeffs])
    E = energies / (energies.sum()+1e-12)
    h_idx = np.linspace(1, K, len(E), dtype=int).clip(1, K)
    mag_wt = mag_r[h_idx] / (mag_r[1:len(E)+1].sum()+1e-12)
    Omega = E * mag_wt + 1e-12
    Omega /= Omega.sum()
    xi = np.linspace(0,1,len(Omega))
    xo = np.linspace(0,1,n_wb)
    Omega_w = interp1d(xi, Omega, kind='linear')(xo)
    Omega_w = np.maximum(Omega_w,0)
    Omega_w /= Omega_w.sum()+1e-12
    r_stats = []
    for sigma in [1,2,4,8]:
        rs = ndimage.gaussian_filter1d(r, sigma, mode='wrap')
        rm = rs.mean()
        rs_ = rs.std()
        r_stats.extend([rm, rs_, float(rs.max()-rs.min()),
                        float(np.percentile(rs,75)-np.percentile(rs,25)),
                        float(scipy.stats.skew(rs)),
                        float(np.sum(rs>rm)/len(rs)),
                        float(np.percentile(rs,90)-np.percentile(rs,10)),
                        float(np.var(rs)/(rm**2+1e-12)),
                        float(np.sum(np.abs(np.diff(rs)))/(len(rs)+1e-12)),
                        float(np.max(rs)/(rm+1e-12))])
    r_stats = np.array(r_stats[:40])
    feat = np.concatenate([fd_r, Omega_w, r_stats])
    ratios = fd_r[1:21] / (fd_r[:20]+1e-12)
    feat = np.concatenate([feat, ratios])
    assert len(feat)==160, f'C1 dim {len(feat)}'
    return feat

print(f'C1: {len(c1_apcfw_plus(t_cnt))}-d (expected 160)')


C1: 160-d (expected 160)


## Cell 8 — AMST C2: Topological Persistence via Ripser (90-d)
Uses Vietoris-Rips persistence on contour point cloud for topological features.


In [8]:
def c2_topological(cnt, n_sample=100, k_lifetimes=15):
    N = len(cnt)
    if N > n_sample:
        idx = np.linspace(0, N-1, n_sample, dtype=int)
        pts = cnt[idx]
    else:
        pts = cnt.copy()
    pts -= pts.mean(axis=0)
    s = np.sqrt((pts**2).sum(axis=1)).max()
    if s > 0: pts /= s
    try:
        dgms = ripser(pts, maxdim=1)['dgms']
    except Exception:
        return np.zeros(90)
    def vec(dgm, k=k_lifetimes):
        fin = dgm[dgm[:,1] < np.inf]
        if len(fin)==0:
            return np.zeros(k), np.zeros(k), np.zeros(8)
        lt = np.sort(fin[:,1]-fin[:,0])[::-1]
        bt = np.sort(fin[:,0])
        lt_v = np.zeros(k)
        lt_v[:min(len(lt),k)] = lt[:k]
        bt_v = np.zeros(k)
        bt_v[:min(len(bt),k)] = bt[:k]
        tot = lt.sum()+1e-12
        mx = lt[0] if len(lt)>0 else 0.0
        betti = float((lt>0.01).sum())
        ent = -np.sum(lt/tot * np.log(lt/tot+1e-12))
        med = float(np.median(lt)) if len(lt)>0 else 0.0
        var = float(np.var(lt)) if len(lt)>0 else 0.0
        n_bars = float(len(fin))
        mean_lt = float(lt.mean()) if len(lt)>0 else 0.0
        return lt_v, bt_v, np.array([tot, mx, betti, ent, med, var, n_bars, mean_lt])
    lt0, bt0, st0 = vec(dgms[0])
    lt1, bt1, st1 = vec(dgms[1])
    feat = np.concatenate([lt0, lt1, bt0[:6], bt1[:6], st0, st1])
    out = np.zeros(90)
    out[:min(len(feat),90)] = feat[:90]
    return out

print(f'C2: {len(c2_topological(t_cnt))}-d (expected 90)')


C2: 90-d (expected 90)


## Cell 9 — AMST C3: SPD Riemannian Manifold (210-d)


In [9]:
def c3_spd(bw, d=20):
    img = bw.astype(float)
    rows = []
    for sigma in [1,2,4,8]:
        g = ndimage.gaussian_filter(img, sigma)
        gx = ndimage.sobel(g, axis=1)
        gy = ndimage.sobel(g, axis=0)
        mag = np.sqrt(gx**2+gy**2)
        lap = ndimage.laplace(g)
        rows.extend([g.flatten(), gx.flatten(), gy.flatten(), mag.flatten(), lap.flatten()])
    fm = np.array(rows[:d], dtype=float)
    fm -= fm.mean(axis=1, keepdims=True)
    fm /= np.linalg.norm(fm, axis=1, keepdims=True) + 1e-12
    S = (fm @ fm.T) / (fm.shape[1]-1) + 1e-5*np.eye(d)
    ev, evec = np.linalg.eigh(S)
    ev = np.maximum(ev, 1e-10)
    logS = evec @ np.diag(np.log(ev)) @ evec.T
    return logS[np.triu_indices(d)]

print(f'C3: {len(c3_spd(t_bw))}-d (expected 210)')


C3: 210-d (expected 210)


## Cell 10 — AMST C4: Multi-Scale Morphological Profile (128-d)
Extracts Euler number, compactness, aspect ratio, rectangularity, and circularity for global shape description.


In [10]:
def c4_morphological(bw):
    feats = []
    area0 = float(bw.sum()) + 1e-12
    for r in range(1, 17):
        feats.append(opening(bw>0, disk(r)).sum() / area0)
    for r in range(1, 17):
        feats.append(closing(bw>0, disk(r)).sum() / area0)
    dt = ndimage.distance_transform_edt(bw>0)
    hist, _ = np.histogram(dt.flatten(), bins=28, range=(0, dt.max()+1e-8), density=True)
    feats.extend(hist.tolist())
    try:
        skel = skeletonize(bw>0)
        sk_a = skel.sum()
        from scipy.ndimage import uniform_filter as uf
        n3 = uf(skel.astype(float), size=3) * 9
        ep = ((n3==2) & skel).sum()
        br = ((n3>=4) & skel).sum()
        feats.extend([sk_a/(area0+1e-12), ep/(sk_a+1e-12), br/(sk_a+1e-12),
                      float(ep), float(br),
                      float(np.mean(dt[bw>0]))/(dt.max()+1e-12),
                      float(np.std(dt[bw>0]))/(dt.max()+1e-12),
                      float(np.max(dt)) / (min(bw.shape)+1e-12)])
    except Exception:
        feats.extend([0.0]*8)
    h,w = bw.shape
    cy, cx = h/2, w/2
    yg, xg = np.mgrid[0:h, 0:w]
    rmap = np.sqrt((xg-cx)**2 + (yg-cy)**2)
    bins = np.linspace(0, rmap.max()+1e-8, 17)
    for b0, b1 in zip(bins[:-1], bins[1:]):
        ring = (rmap>=b0) & (rmap<b1)
        feats.append(((ring) & (bw>0)).sum() / (ring.sum()+1e-12))
    try:
        lbp = local_binary_pattern(bw.astype(np.uint8)*255, P=8, R=1, method='uniform')
        lh, _ = np.histogram(lbp.flatten(), bins=16, range=(0,16), density=True)
        feats.extend(lh.tolist())
    except Exception:
        feats.extend([0.0]*16)
    for sc in [4,8,16,32,48,64,96,112]:
        sm = cv2.resize(bw.astype(np.uint8), (sc,sc), interpolation=cv2.INTER_NEAREST)
        feats.append(sm.sum() / (sc**2+1e-12))
    try:
        from skimage.measure import euler_number
        e4 = euler_number(bw>0, connectivity=1)
        e8 = euler_number(bw>0, connectivity=2)
        feats.extend([float(e4), float(e8), float(abs(e4-e8)), float(e4/(area0**0.5+1e-12))])
    except Exception:
        feats.extend([0.0]*4)
    cnts = find_contours(bw.astype(float), 0.5)
    if cnts:
        c = max(cnts, key=len)
        perim = len(c)
        compact = perim**2 / (4*np.pi*area0+1e-12)
        feats.extend([np.log(compact+1e-10), float(cv2.arcLength(c.astype(np.float32), True)),
                      float(cv2.contourArea(c.astype(np.float32)))/(area0+1e-12),
                      float(np.sqrt(area0)/(perim+1e-12)),
                      float(perim)/(h+w+1e-12), float(area0)/(h*w+1e-12)])
    else:
        feats.extend([0.0]*6)
    while len(feats) < 128:
        feats.append(0.0)
    return np.array(feats[:128])

print(f'C4: {len(c4_morphological(t_bw))}-d (expected 128)')


C4: 128-d (expected 128)


## Cell 11 — AMST C5: Shape Complexity & Moment Invariants (30-d)
Computes fractal dimension, LZ complexity, and Hu moments for high-level shape characterization.


In [11]:
def c5_complexity(bw, cnt):
    feats = []
    area = float(bw.sum()) + 1e-12
    perim = float(len(cnt))
    compact = perim**2 / (4*np.pi*area)
    feats.append(np.log(compact+1e-10))
    feats.append(area / (IMG_SIZE[0]*IMG_SIZE[1]))
    feats.append(perim / (4*IMG_SIZE[0]))
    try:
        hull = ConvexHull(cnt)
        feats.append(area / (hull.volume+1e-12))
        feats.append(hull.area / (perim+1e-12))
    except Exception:
        feats.extend([0.0, 0.0])
    ev_cnt = np.linalg.eigvalsh(np.cov(cnt.T))
    feats.append(np.sort(ev_cnt)[::-1][0] / (np.sort(ev_cnt)[::-1][1]+1e-12))
    m = cv2.moments(bw.astype(np.uint8))
    hu = cv2.HuMoments(m).flatten()
    feats.extend(np.sign(hu) * np.log(np.abs(hu)+1e-12))
    r = np.sqrt((cnt**2).sum(axis=1))
    feats.extend([r.mean(), r.std(), r.min(), r.max(),
                  float(np.percentile(r,25)), float(np.percentile(r,75)),
                  float(scipy.stats.skew(r)), float(scipy.stats.kurtosis(r))])
    x_c, y_c = cnt[:,1], cnt[:,0]
    x1=np.gradient(x_c); y1=np.gradient(y_c)
    x2=np.gradient(x1); y2=np.gradient(y1)
    kappa = (x1*y2-x2*y1)/(x1**2+y1**2+1e-12)**1.5
    feats.extend([float(np.mean(np.abs(kappa))), float(np.std(kappa)),
                  float(np.sum(np.diff(np.sign(kappa))!=0)),
                  float(np.max(np.abs(kappa))),
                  float(np.percentile(np.abs(kappa),90)),
                  float(scipy.stats.entropy(np.abs(kappa)/(np.abs(kappa).sum()+1e-12)+1e-12))])
    try:
        sizes = np.arange(2, min(bw.shape)//4, 2)
        counts = []
        for s in sizes:
            reduced = bw[::s, ::s]
            counts.append(float(reduced.sum()))
        counts = np.array(counts)
        if len(counts) > 2 and counts[-1] > 0:
            coeffs = np.polyfit(np.log(sizes[:len(counts)]), np.log(counts+1e-12), 1)
            fd = -coeffs[0]
        else:
            fd = 0.0
        feats.append(fd)
        feats.append(fd / 2.0)
    except Exception:
        feats.extend([0.0, 0.0])
    while len(feats) < 30:
        feats.append(0.0)
    return np.array(feats[:30])

print(f'C5: {len(c5_complexity(t_bw, t_cnt))}-d (expected 30)')


C5: 30-d (expected 30)


## Cell 12 — Full AMST Feature Extraction


In [12]:
def amst_descriptor(bw, cnt):
    return np.concatenate([c1_apcfw_plus(cnt), c2_topological(cnt),
                           c3_spd(bw), c4_morphological(bw), c5_complexity(bw, cnt)])

FEAT_CACHE = 'mpeg7_features_v2.npz'
if Path(FEAT_CACHE).exists():
    print('Loading feature cache...')
    fc = np.load(FEAT_CACHE, allow_pickle=True)
    X_hog = fc['X_hog']; X_zern = fc['X_zern']; X_four = fc['X_four']
    X_wav = fc['X_wav']; X_css = fc['X_css']; X_sc = fc['X_sc']
    X_amst = fc['X_amst']
    ext_times = fc['extraction_times'].item() if 'extraction_times' in fc else {}
    print('Cache loaded.')
else:
    N = len(all_images)
    hog_l=[]; zern_l=[]; four_l=[]; wav_l=[]; css_l=[]; sc_l=[]; amst_l=[]
    ext_times = {}
    t0=time.time()
    for i in tqdm(range(N), desc='HOG'):
        try: hog_l.append(hog_descriptor(all_images[i]))
        except: hog_l.append(np.zeros(324))
    ext_times['HOG'] = time.time()-t0
    t0=time.time()
    for i in tqdm(range(N), desc='Zernike'):
        try: zern_l.append(zernike_descriptor(all_images[i]))
        except: zern_l.append(np.zeros(36))
    ext_times['Zernike'] = time.time()-t0
    t0=time.time()
    for i in tqdm(range(N), desc='Fourier'):
        try: four_l.append(fourier_descriptor(all_contours[i]))
        except: four_l.append(np.zeros(39))
    ext_times['Fourier'] = time.time()-t0
    t0=time.time()
    for i in tqdm(range(N), desc='Wavelet'):
        try: wav_l.append(wavelet_descriptor(all_contours[i]))
        except: wav_l.append(np.zeros(5))
    ext_times['Wavelet'] = time.time()-t0
    t0=time.time()
    for i in tqdm(range(N), desc='CSS'):
        try: css_l.append(css_descriptor(all_contours[i]))
        except: css_l.append(np.zeros(12))
    ext_times['CSS'] = time.time()-t0
    t0=time.time()
    for i in tqdm(range(N), desc='SC'):
        try: sc_l.append(shape_context(all_contours[i]))
        except: sc_l.append(np.zeros(60))
    ext_times['SC'] = time.time()-t0
    t0=time.time()
    for i in tqdm(range(N), desc='AMST'):
        try: amst_l.append(amst_descriptor(all_images[i], all_contours[i]))
        except: amst_l.append(np.zeros(618))
    ext_times['AMST'] = time.time()-t0
    X_hog = np.nan_to_num(np.array(hog_l))
    X_zern = np.nan_to_num(np.array(zern_l))
    X_four = np.nan_to_num(np.array(four_l))
    X_wav = np.nan_to_num(np.array(wav_l))
    X_css = np.nan_to_num(np.array(css_l))
    X_sc = np.nan_to_num(np.array(sc_l))
    X_amst = np.nan_to_num(np.array(amst_l))
    np.savez_compressed(FEAT_CACHE, X_hog=X_hog, X_zern=X_zern, X_four=X_four,
                        X_wav=X_wav, X_css=X_css, X_sc=X_sc, X_amst=X_amst,
                        extraction_times=ext_times)

print(f'Feature shapes: HOG{X_hog.shape} AMST{X_amst.shape} y{y_mpeg.shape}')
assert X_amst.shape[1]==618


AMST: 100%|██████████| 1400/1400 [10:26<00:00,  2.23it/s]


Feature shapes: HOG(1400, 324) AMST(1400, 618) y(1400,)


## Cell 13 — Kimia216 Feature Extraction


In [13]:
KIMIA_FEAT_CACHE = 'kimia216_features.npz'
if Path(KIMIA_FEAT_CACHE).exists():
    kf = np.load(KIMIA_FEAT_CACHE, allow_pickle=True)
    X_kim_amst = kf['X_amst']
    X_kim_hog = kf['X_hog']
    print(f'Loaded Kimia features: AMST{X_kim_amst.shape}')
else:
    Nk = len(kim_images)
    amst_l=[]; hog_l=[]
    for i in tqdm(range(Nk), desc='Kimia AMST'):
        try: amst_l.append(amst_descriptor(kim_images[i], kim_contours[i]))
        except: amst_l.append(np.zeros(618))
    for i in tqdm(range(Nk), desc='Kimia HOG'):
        try: hog_l.append(hog_descriptor(kim_images[i]))
        except: hog_l.append(np.zeros(324))
    X_kim_amst = np.nan_to_num(np.array(amst_l))
    X_kim_hog = np.nan_to_num(np.array(hog_l))
    np.savez_compressed(KIMIA_FEAT_CACHE, X_amst=X_kim_amst, X_hog=X_kim_hog)
print(f'Kimia features: AMST{X_kim_amst.shape}, HOG{X_kim_hog.shape}')


Kimia HOG: 100%|██████████| 216/216 [00:00<00:00, 733.59it/s]


Kimia features: AMST(216, 618), HOG(216, 324)


## Cell 14 — Deep Learning Features (ViT-B/16 + ResNet50 + EfficientNet-B0)
Extracts patch-level features using ViT-B/16 transformer (timm) for deep visual representation.


In [14]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

DL_CACHE = 'mpeg7_dl_features.npz'
if Path(DL_CACHE).exists():
    dl = np.load(DL_CACHE, allow_pickle=True)
    X_vit = dl['X_vit']
    X_resnet = dl['X_resnet']
    X_effnet = dl['X_effnet']
    print(f'DL features loaded: ViT{X_vit.shape}, ResNet{X_resnet.shape}, EffNet{X_effnet.shape}')
else:
    tfms = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ])
    def img_to_tensor(bw):
        rgb = np.stack([bw]*3, axis=-1).astype(np.float32)
        img = transforms.ToPILImage()(rgb)
        return tfms(img).unsqueeze(0)
    print('Loading ViT-B/16...')
    vit = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0).to(DEVICE).eval()
    vit_feats = []
    for i in tqdm(range(len(all_images)), desc='ViT'):
        with torch.no_grad():
            x = img_to_tensor(all_images[i]).to(DEVICE)
            vit_feats.append(vit(x).cpu().numpy().flatten())
    X_vit = np.nan_to_num(np.array(vit_feats))
    del vit
    print('Loading ResNet50...')
    resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    resnet = nn.Sequential(*list(resnet.children())[:-1]).to(DEVICE).eval()
    res_feats = []
    for i in tqdm(range(len(all_images)), desc='ResNet'):
        with torch.no_grad():
            x = img_to_tensor(all_images[i]).to(DEVICE)
            res_feats.append(resnet(x).cpu().numpy().flatten())
    X_resnet = np.nan_to_num(np.array(res_feats))
    del resnet
    print('Loading EfficientNet-B0...')
    effnet = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0).to(DEVICE).eval()
    eff_feats = []
    for i in tqdm(range(len(all_images)), desc='EfficientNet'):
        with torch.no_grad():
            x = img_to_tensor(all_images[i]).to(DEVICE)
            eff_feats.append(effnet(x).cpu().numpy().flatten())
    X_effnet = np.nan_to_num(np.array(eff_feats))
    del effnet
    np.savez_compressed(DL_CACHE, X_vit=X_vit, X_resnet=X_resnet, X_effnet=X_effnet)
print(f'ViT: {X_vit.shape[1]}-d, ResNet: {X_resnet.shape[1]}-d, EffNet: {X_effnet.shape[1]}-d')


Using device: cpu
Loading ViT-B/16...


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

ViT: 100%|██████████| 1400/1400 [05:30<00:00,  4.23it/s]


Loading ResNet50...
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 387MB/s]
ResNet: 100%|██████████| 1400/1400 [02:35<00:00,  9.00it/s]


Loading EfficientNet-B0...


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

EfficientNet: 100%|██████████| 1400/1400 [01:20<00:00, 17.39it/s]


ViT: 768-d, ResNet: 2048-d, EffNet: 1280-d


## Cell 15 — Kimia216 Deep Features


In [15]:
KIMIA_DL_CACHE = 'kimia216_dl_features.npz'
if Path(KIMIA_DL_CACHE).exists():
    kd = np.load(KIMIA_DL_CACHE, allow_pickle=True)
    X_kim_vit = kd['X_vit']
    X_kim_resnet = kd['X_resnet']
    X_kim_effnet = kd['X_effnet']
    print(f'Kimia DL: ViT{X_kim_vit.shape}')
else:
    def batch_extract(model, desc='Extract'):
        feats = []
        for i in tqdm(range(len(kim_images)), desc=desc):
            with torch.no_grad():
                x = img_to_tensor(kim_images[i]).to(DEVICE)
                feats.append(model(x).cpu().numpy().flatten())
        return np.nan_to_num(np.array(feats))
    vit2 = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0).to(DEVICE).eval()
    X_kim_vit = batch_extract(vit2, 'Kimia ViT')
    del vit2
    res2 = nn.Sequential(*list(models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1).children())[:-1]).to(DEVICE).eval()
    X_kim_resnet = batch_extract(res2, 'Kimia ResNet')
    del res2
    eff2 = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0).to(DEVICE).eval()
    X_kim_effnet = batch_extract(eff2, 'Kimia EffNet')
    del eff2
    np.savez_compressed(KIMIA_DL_CACHE, X_vit=X_kim_vit, X_resnet=X_kim_resnet, X_effnet=X_kim_effnet)
print(f'Kimia ViT: {X_kim_vit.shape}, ResNet: {X_kim_resnet.shape}, EffNet: {X_kim_effnet.shape}')


Kimia EffNet: 100%|██████████| 216/216 [00:12<00:00, 17.50it/s]


Kimia ViT: (216, 768), ResNet: (216, 2048), EffNet: (216, 1280)


## Cell 16 — Combined Feature Space & Evaluation Setup
Stacks AMST (618-d) + ViT (768-d) + HOG (324-d) with per-component normalization.


In [16]:
def build_combined(X_amst, X_vit, X_hog):
    return np.concatenate([
        np.nan_to_num(X_amst),
        np.nan_to_num(X_vit),
        np.nan_to_num(X_hog),
    ], axis=1)

def amst_component_dims():
    return {'C1':160, 'C2':90, 'C3':210, 'C4':128, 'C5':30}

COMP_DIMS = list(amst_component_dims().values())
COMBINED_COMP_DIMS = COMP_DIMS + [768, 324]
COMBINED_NAMES = list(amst_component_dims().keys()) + ['ViT', 'HOG']

# Ensure Kimia features are loaded
KIMIA_FEAT_CACHE = 'kimia216_features.npz'
if Path(KIMIA_FEAT_CACHE).exists():
    kf = np.load(KIMIA_FEAT_CACHE, allow_pickle=True)
    X_kim_amst = kf['X_amst']
    X_kim_hog = kf['X_hog']
else:
    print(f'Warning: Kimia feature cache {KIMIA_FEAT_CACHE} not found. Ensure Cell 13 was run.')
    X_kim_amst = np.zeros((len(kim_images), 618))
    X_kim_hog = np.zeros((len(kim_images), 324))

KIMIA_DL_CACHE = 'kimia216_dl_features.npz'
if Path(KIMIA_DL_CACHE).exists():
    kd = np.load(KIMIA_DL_CACHE, allow_pickle=True)
    X_kim_vit = kd['X_vit']
else:
    print(f'Warning: Kimia deep learning feature cache {KIMIA_DL_CACHE} not found. Ensure Cell 15 was run.')
    X_kim_vit = np.zeros((len(kim_images), 768))

X_comb_mpeg = build_combined(X_amst, X_vit, X_hog)
X_comb_kimia = build_combined(X_kim_amst, X_kim_vit, X_kim_hog)

print(f'MPEG-7 combined: {X_comb_mpeg.shape}')
print(f'Kimia combined:  {X_comb_kimia.shape}')
print(f'Components: {list(zip(COMBINED_NAMES, COMBINED_COMP_DIMS))}')

MPEG-7 combined: (1400, 1710)
Kimia combined:  (216, 1710)
Components: [('C1', 160), ('C2', 90), ('C3', 210), ('C4', 128), ('C5', 30), ('ViT', 768), ('HOG', 324)]


## Cell 17 — 10-Fold Cross-Validation
10-fold stratified CV using SVM with per-component normalization and Fisher feature selection.


In [17]:
N_FOLDS = 10

def per_component_normalize(X_raw, comp_dims, train_idx, test_idx):
    X_tr_n = np.zeros_like(X_raw[train_idx])
    X_te_n = np.zeros_like(X_raw[test_idx])
    start = 0
    for dim in comp_dims:
        end = start+dim
        mu = X_raw[train_idx][:,start:end].mean(axis=0)
        std = X_raw[train_idx][:,start:end].std(axis=0) + 1e-10
        X_tr_n[:,start:end] = (X_raw[train_idx][:,start:end]-mu)/std
        X_te_n[:,start:end] = (X_raw[test_idx][:,start:end]-mu)/std
        start = end
    return X_tr_n, X_te_n

def eval_svm(X_raw, y, k_features=0):
    X_raw = np.nan_to_num(X_raw.copy())
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_accs, fold_f1s = [], []
    all_yt, all_yp = [], []
    for tr, te in skf.split(X_raw, y):
        sc = StandardScaler()
        X_tr = sc.fit_transform(X_raw[tr])
        X_te = sc.transform(X_raw[te])
        if k_features > 0:
            k = min(k_features, X_tr.shape[1])
            sel = SelectKBest(f_classif, k=k)
            X_tr = sel.fit_transform(X_tr, y[tr])
            X_te = sel.transform(X_te)
        clf = SVC(kernel='rbf', C=100, gamma='scale', decision_function_shape='ovr', random_state=SEED)
        clf.fit(X_tr, y[tr])
        yp = clf.predict(X_te)
        fold_accs.append(accuracy_score(y[te], yp))
        fold_f1s.append(f1_score(y[te], yp, average='macro'))
        all_yt.extend(y[te].tolist())
        all_yp.extend(yp.tolist())
    return fold_accs, np.array(all_yt), np.array(all_yp), fold_f1s

def eval_amst(X_raw, y, comp_dims, k_features=350):
    X_raw = np.nan_to_num(X_raw.copy())
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_accs, fold_f1s = [], []
    all_yt, all_yp = [], []
    for tr, te in skf.split(X_raw, y):
        X_tr_n, X_te_n = per_component_normalize(X_raw, comp_dims, tr, te)
        k = min(k_features, X_tr_n.shape[1])
        sel = SelectKBest(f_classif, k=k)
        X_tr_s = sel.fit_transform(X_tr_n, y[tr])
        X_te_s = sel.transform(X_te_n)
        sc = StandardScaler()
        X_tr = sc.fit_transform(X_tr_s)
        X_te = sc.transform(X_te_s)
        clf = SVC(kernel='rbf', C=100, gamma='scale', decision_function_shape='ovr', random_state=SEED)
        clf.fit(X_tr, y[tr])
        yp = clf.predict(X_te)
        fold_accs.append(accuracy_score(y[te], yp))
        fold_f1s.append(f1_score(y[te], yp, average='macro'))
        all_yt.extend(y[te].tolist())
        all_yp.extend(yp.tolist())
    return fold_accs, np.array(all_yt), np.array(all_yp), fold_f1s

results = {}
preds = {}
print('Running 10-fold CV evaluations\n')

print('1/4  HOG (SVM-RBF)...')
fa, yt, yp, f1s = eval_svm(X_hog, y_mpeg)
results['HOG'] = {'accs':fa, 'mean':np.mean(fa)*100, 'std':np.std(fa)*100, 'f1s':f1s}

print('2/4  AMST (per-component + Fisher + SVM)...')
fa, yt, yp, f1s = eval_amst(X_amst, y_mpeg, COMP_DIMS, k_features=350)
results['AMST'] = {'accs':fa, 'mean':np.mean(fa)*100, 'std':np.std(fa)*100, 'f1s':f1s}
print(f'  Accuracy: {np.mean(fa)*100:.2f}%')

print('3/4  ViT-B/16 (SVM-RBF)...')
fa, yt, yp, f1s = eval_svm(X_vit, y_mpeg)
results['ViT-B/16'] = {'accs':fa, 'mean':np.mean(fa)*100, 'std':np.std(fa)*100, 'f1s':f1s}

print('4/4  AMST+ViT+HOG (SVM, 1000 Fisher features)...')
fa, yt, yp, f1s = eval_amst(X_comb_mpeg, y_mpeg, COMBINED_COMP_DIMS, k_features=1000)
results['AMST+ViT+HOG'] = {'accs':fa, 'mean':np.mean(fa)*100, 'std':np.std(fa)*100, 'f1s':f1s}
preds['AMST+ViT+HOG'] = (yt, yp)
print(f'  Accuracy: {np.mean(fa)*100:.2f}%')

print('\n' + '='*70)
print(f"{'Method':<40} {'Accuracy':>10} {'Std':>8}")
print('='*70)
best_method = max(results, key=lambda n:results[n]['mean'])
for nm in sorted(results, key=lambda n:results[n]['mean'], reverse=True):
    marker = ' *' if nm == best_method else '  '
    print(f"{marker} {nm:<38} {results[nm]['mean']:>8.2f}% +/-{results[nm]['std']:>5.2f}%")
print('='*70)
best_acc_val = results[best_method]['mean']
print(f'\nBest accuracy: {best_acc_val:.2f}%')


Running 10-fold CV evaluations

1/4  HOG (SVM-RBF)...
2/4  AMST (per-component + Fisher + SVM)...
  Accuracy: 92.14%
3/4  ViT-B/16 (SVM-RBF)...
4/4  AMST+ViT+HOG (SVM, 1000 Fisher features)...
  Accuracy: 96.36%

Method                                     Accuracy      Std
 * AMST+ViT+HOG                              96.36% +/- 1.58%
   ViT-B/16                                  94.29% +/- 1.94%
   AMST                                      92.14% +/- 1.89%
   HOG                                       89.79% +/- 2.73%

Best accuracy: 96.36%


## Cell 18 — Ablation Study (Component Contribution)
Ablation study using 10-fold CV with SVM to measure the contribution of each AMST component.


In [18]:
ablation_results = {}
ablation_configs = {
    'C1: APCFW+': [160],
    'C1+C3: +SPD': [160, 210],
    'C1-C3: +Topology': [160, 90, 210],
    'C1-C4: +Morphology': [160, 90, 210, 128],
    'Full AMST': [160, 90, 210, 128, 30],
}
print('Ablation study:')
for name, comp_dims in ablation_configs.items():
    start = 0
    total_d = sum(comp_dims)
    X_sub = np.zeros((len(all_images), total_d))
    for dim in comp_dims:
        end = start + dim
        X_sub[:, start:end] = X_amst[:, start:end]
        start = end
    fa, yt, yp, f1s = eval_amst(X_sub, y_mpeg, comp_dims, k_features=min(350, total_d))
    ablation_results[name] = {'mean':np.mean(fa)*100, 'std':np.std(fa)*100}
    print(f'  {name:30s}: {np.mean(fa)*100:6.2f}% +/-{np.std(fa)*100:4.2f}%')
print(f'\nComponent contribution gains:')
prev_acc = 0
for nm in ablation_configs:
    cur = ablation_results[nm]['mean']
    if prev_acc > 0:
        print(f'  +{nm.split("+")[-1].strip():20s}: {cur-prev_acc:+.2f}pp')
    else:
        print(f'  {nm:28s}: {cur:.2f}% (baseline)')
    prev_acc = cur


Ablation study:
  C1: APCFW+                    :  65.43% +/-1.81%
  C1+C3: +SPD                   :  85.79% +/-2.87%
  C1-C3: +Topology              :  87.93% +/-2.18%
  C1-C4: +Morphology            :  91.71% +/-1.90%
  Full AMST                     :  92.14% +/-1.89%

Component contribution gains:
  C1: APCFW+                  : 65.43% (baseline)
  +SPD                 : +20.36pp
  +Topology            : +2.14pp
  +Morphology          : +3.79pp
  +Full AMST           : +0.43pp


## Cell 19 — Hyperparameter Sensitivity Analysis

The SVM hyperparameters used throughout this study (C=100, gamma='scale') are shared across all methods to keep the comparison uniform. This cell verifies that choice with a grid search over C and gamma on the combined AMST+ViT+HOG feature space, using a single stratified train/held-out split (rather than a full nested cross-validation) to keep runtime modest on a CPU-only environment. The held-out accuracy of the searched best configuration is compared against the accuracy obtained with the fixed default (C=100, gamma='scale') used elsewhere in this notebook.


In [19]:
from sklearn.model_selection import GridSearchCV, train_test_split

grid_C = [1, 10, 50, 100, 200, 500]
grid_gamma = ['scale', 'auto']

X_hp = np.nan_to_num(X_comb_mpeg.copy())
tr_idx, te_idx = train_test_split(np.arange(len(y_mpeg)), test_size=0.2,
                                   stratify=y_mpeg, random_state=SEED)
Xtr_n, Xte_n = per_component_normalize(X_hp, COMBINED_COMP_DIMS, tr_idx, te_idx)
sel_hp = SelectKBest(f_classif, k=min(1000, Xtr_n.shape[1]))
Xtr_s = sel_hp.fit_transform(Xtr_n, y_mpeg[tr_idx])
Xte_s = sel_hp.transform(Xte_n)
sc_hp = StandardScaler()
Xtr_f = sc_hp.fit_transform(Xtr_s)
Xte_f = sc_hp.transform(Xte_s)

param_grid = {'C': grid_C, 'gamma': grid_gamma}
gs = GridSearchCV(SVC(kernel='rbf', decision_function_shape='ovr', random_state=SEED),
                   param_grid, cv=3, n_jobs=-1)
gs.fit(Xtr_f, y_mpeg[tr_idx])
searched_test_acc = gs.score(Xte_f, y_mpeg[te_idx]) * 100

default_clf = SVC(kernel='rbf', C=100, gamma='scale', decision_function_shape='ovr', random_state=SEED)
default_clf.fit(Xtr_f, y_mpeg[tr_idx])
default_test_acc = default_clf.score(Xte_f, y_mpeg[te_idx]) * 100

print('Hyperparameter grid search (combined AMST+ViT+HOG features):')
print(f'  Best params found: {gs.best_params_}  (3-fold CV accuracy: {gs.best_score_*100:.2f}%)')
print(f'  Held-out accuracy with searched best params: {searched_test_acc:.2f}%')
print(f'  Held-out accuracy with fixed default (C=100, gamma=scale): {default_test_acc:.2f}%')
print(f'  Difference: {searched_test_acc - default_test_acc:+.2f}pp')

pd.DataFrame(gs.cv_results_).to_csv('hyperparameter_grid_search.csv', index=False)
print('\\nFull grid search results saved to hyperparameter_grid_search.csv')


Hyperparameter grid search (combined AMST+ViT+HOG features):
  Best params found: {'C': 10, 'gamma': 'scale'}  (3-fold CV accuracy: 93.93%)
  Held-out accuracy with searched best params: 96.79%
  Held-out accuracy with fixed default (C=100, gamma=scale): 96.79%
  Difference: +0.00pp
\nFull grid search results saved to hyperparameter_grid_search.csv


## Figure 1 — MPEG-7 Dataset Sample Silhouettes


In [20]:
fig, axes = plt.subplots(5, 10, figsize=(20, 10))
classes = sorted(cc.keys())
class_to_first = {}
for i, lbl in enumerate(labels_lower):
    if lbl not in class_to_first:
        class_to_first[lbl] = i
for i, cls in enumerate(classes[:50]):
    idx = class_to_first[cls]
    ax = axes[i//10, i%10]
    ax.imshow(all_images[idx], cmap='gray')
    ax.set_title(cls, fontsize=8)
    ax.axis('off')
plt.suptitle('MPEG-7 CE-Shape-1 Part B — Sample Shapes', fontsize=16)
plt.tight_layout()
plt.savefig('fig1_mpeg7_dataset.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')


Figure 1 saved.


## Figure 2 — Accuracy Comparison Bar Chart


In [21]:
sns.set_style('whitegrid')
methods = list(results.keys())
means = [results[m]['mean'] for m in methods]
stds = [results[m]['std'] for m in methods]
colors_bar = ['#e74c3c' if m == 'AMST+ViT+HOG' else '#3498db' for m in methods]
fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.bar(methods, means, yerr=stds, capsize=6, color=colors_bar, edgecolor='black', linewidth=1.2, width=0.6)
for bar, m, v in zip(bars, methods, means):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.2, f'{v:.2f}%',
            ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylabel('10-Fold CV Accuracy (%)', fontsize=14, fontweight='bold')
ax.set_title('Shape Classification Accuracy on MPEG-7 CE-Shape-1', fontsize=15, fontweight='bold')
ax.set_ylim(80, 100)
ax.set_xticks(range(len(methods)))
ax.set_xticklabels(methods, fontsize=11)
ax.axhline(y=96, color='green', linestyle='--', linewidth=1.5, alpha=0.7, label='96% Threshold')
ax.legend(fontsize=10)
plt.grid(axis='y', alpha=0.3)
sns.despine()
plt.tight_layout()
plt.savefig('fig2_accuracy_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print('Figure 2 saved.')


Figure 2 saved.


## Figure 3 — Confusion Matrix (Best Method)


In [22]:
best_preds = preds[list(preds.keys())[-1]]
cm = confusion_matrix(best_preds[0], best_preds[1])
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, cmap='Blues', annot=False, cbar=True, ax=ax, linewidths=0.3, linecolor='gray')
ax.set_xlabel('Predicted Label', fontsize=14, fontweight='bold')
ax.set_ylabel('True Label', fontsize=14, fontweight='bold')
ax.set_title(f'Confusion Matrix — {list(preds.keys())[-1]} (Accuracy: {np.trace(cm)/len(best_preds[0])*100:.2f}%)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'Figure 3 saved. Correct: {np.trace(cm)}/{len(best_preds[0])}')


Figure 3 saved. Correct: 1349/1400


## Cell 22 — Statistical Significance (Paired t-test + Friedman + Nemenyi)

Significance of the accuracy differences across methods is assessed with three complementary tests. The Friedman test with Nemenyi post-hoc analysis is treated as the primary evidence of significance, since it operates on the joint rank structure across all methods and folds rather than pairwise comparisons. Paired t-tests and Wilcoxon signed-rank tests are reported alongside as standard descriptive statistics; because the ten per-fold accuracies come from a shared cross-validation split rather than fully independent samples, they are interpreted as supporting rather than sole evidence of significance.


In [23]:
from scipy.stats import friedmanchisquare, wilcoxon, ttest_rel
import scikit_posthocs as sp

all_fold_accs = {}
for nm, r in results.items():
    if 'accs' in r:
        all_fold_accs[nm] = r['accs']

methods_list = list(all_fold_accs.keys())
print(f'\nPaired t-tests (10-fold CV):')
for i in range(len(methods_list)):
    for j in range(i+1, len(methods_list)):
        a, b = methods_list[i], methods_list[j]
        t_stat, t_p = ttest_rel(all_fold_accs[a], all_fold_accs[b])
        w_stat, w_p = wilcoxon(all_fold_accs[a], all_fold_accs[b])
        sig = 'SIGNIFICANT' if t_p < 0.05 else 'not significant'
        print(f'  {a:35s} vs {b:35s}: t-test p={t_p:.4f} ({sig})')

print(f'\nFriedman Test:')
fold_matrix = np.array([all_fold_accs[m] for m in methods_list])
friedman_stat, friedman_p = friedmanchisquare(*[all_fold_accs[m] for m in methods_list])
print(f'  chi2={friedman_stat:.4f}, p={friedman_p:.4f}')
print(f'  Significant differences: {friedman_p < 0.05}')

if friedman_p < 0.05 and len(methods_list) >= 3:
    try:
        nemenyi = sp.posthoc_nemenyi_friedman(np.array([all_fold_accs[m] for m in methods_list]).T)
        nemenyi.index = methods_list
        nemenyi.columns = methods_list
        print(f'\nNemenyi Post-hoc (p-values):')
        print(nemenyi.to_string(float_format='{:.4f}'.format))
        print(f'\nSignificant at 0.05:')
        for i in range(len(methods_list)):
            for j in range(i+1, len(methods_list)):
                if nemenyi.iloc[i,j] < 0.05:
                    print(f'  {methods_list[i]:35s} vs {methods_list[j]:35s}: p={nemenyi.iloc[i,j]:.4f} *')
    except Exception as e:
        print(f'Nemenyi failed: {e}')
        print('Using multiple Wilcoxon with Bonferroni correction...')
        n_tests = len(methods_list)*(len(methods_list)-1)//2
        for i in range(len(methods_list)):
            for j in range(i+1, len(methods_list)):
                _, wp = wilcoxon(all_fold_accs[methods_list[i]], all_fold_accs[methods_list[j]])
                print(f'  {methods_list[i]:35s} vs {methods_list[j]:35s}: Wilcoxon p={wp:.4f} (Bonf: {wp*n_tests:.4f})')



Paired t-tests (10-fold CV):
  HOG                                 vs AMST                               : t-test p=0.0591 (not significant)
  HOG                                 vs ViT-B/16                           : t-test p=0.0011 (SIGNIFICANT)
  HOG                                 vs AMST+ViT+HOG                       : t-test p=0.0000 (SIGNIFICANT)
  AMST                                vs ViT-B/16                           : t-test p=0.0653 (not significant)
  AMST                                vs AMST+ViT+HOG                       : t-test p=0.0003 (SIGNIFICANT)
  ViT-B/16                            vs AMST+ViT+HOG                       : t-test p=0.0043 (SIGNIFICANT)

Friedman Test:
  chi2=21.4062, p=0.0001
  Significant differences: True

Nemenyi Post-hoc (p-values):
                HOG   AMST  ViT-B/16  AMST+ViT+HOG
HOG          1.0000 0.7763    0.0725        0.0001
AMST         0.7763 1.0000    0.4544        0.0055
ViT-B/16     0.0725 0.4544    1.0000        0.2644
AMST+Vi

## Cell 23 — Shape Retrieval with Per-Component Rank Fusion

Evaluates shape retrieval using per-component Z-score normalization followed by cosine similarity and Borda count rank fusion. This approach respects the heterogeneous nature of multi-component descriptors (AMST, combined), where each component captures fundamentally different shape properties with distinct statistical distributions.


In [24]:
def retrieval_map(X, y, metric='cosine'):
    X = np.nan_to_num(X.copy())
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    N = len(y)
    n_q = min(N, 500)
    q_idx = np.random.RandomState(SEED).choice(N, n_q, replace=False)
    APs = []
    for qi in tqdm(q_idx, desc='Retrieval'):
        if metric == 'cosine':
            sim = (Xs @ Xs[qi]) / (np.linalg.norm(Xs, axis=1) * np.linalg.norm(Xs[qi]) + 1e-12)
            ranked = np.argsort(-sim)[1:]
        else:
            dist = np.sqrt(((Xs - Xs[qi])**2).sum(axis=1))
            ranked = np.argsort(dist)[1:]
        rel = (y[ranked] == y[qi]).astype(int)
        if rel.sum() == 0: continue
        cs = np.cumsum(rel)
        pos = np.arange(1, len(ranked)+1)
        APs.append((cs/pos * rel).sum() / rel.sum())
    return float(np.mean(APs)) if APs else 0.0

def retrieval_rank_fusion(X_raw, y, comp_dims, metric='cosine'):
    X_raw = np.nan_to_num(X_raw.copy())
    X_n = np.zeros_like(X_raw)
    start = 0
    for dim in comp_dims:
        end = start+dim
        mu = X_raw[:, start:end].mean(axis=0)
        std = X_raw[:, start:end].std(axis=0) + 1e-10
        X_n[:, start:end] = (X_raw[:, start:end]-mu)/std
        start = end
    N = len(y)
    n_q = min(N, 500)
    q_idx = np.random.RandomState(SEED).choice(N, n_q, replace=False)
    APs = []
    for qi in tqdm(q_idx, desc='Rank fusion'):
        all_ranks = np.zeros((len(comp_dims), N))
        start = 0
        for ci, dim in enumerate(comp_dims):
            end = start+dim
            Xc = X_n[:, start:end]
            if metric == 'cosine':
                sim = (Xc @ Xc[qi]) / (np.linalg.norm(Xc, axis=1) * np.linalg.norm(Xc[qi]) + 1e-12)
                all_ranks[ci] = np.argsort(np.argsort(-sim))
            else:
                dist = np.sqrt(((Xc - Xc[qi])**2).sum(axis=1))
                all_ranks[ci] = np.argsort(np.argsort(dist))
            start = end
        fused_rank = all_ranks.mean(axis=0)
        ranked = np.argsort(fused_rank)
        ranked = ranked[ranked != qi]
        rel = (y[ranked] == y[qi]).astype(int)
        if rel.sum() == 0: continue
        cs = np.cumsum(rel)
        pos = np.arange(1, len(ranked)+1)
        APs.append((cs/pos * rel).sum() / rel.sum())
    return float(np.mean(APs)) if APs else 0.0

retrieval_results = {}
print('\nRetrieval MAP evaluation (global cosine):')
for name, X_ret in [('HOG', X_hog), ('ViT-B/16', X_vit), ('AMST', X_amst), ('AMST+ViT+HOG', X_comb_mpeg)]:
    map_val = retrieval_map(X_ret, y_mpeg, 'cosine')
    retrieval_results[name] = map_val
    print(f'  {name:20s}: MAP = {map_val*100:.2f}%')



Retrieval MAP evaluation (global cosine):


Retrieval: 100%|██████████| 500/500 [00:00<00:00, 3298.38it/s]


  HOG                 : MAP = 54.95%


Retrieval: 100%|██████████| 500/500 [00:03<00:00, 140.13it/s]


  ViT-B/16            : MAP = 63.63%


Retrieval: 100%|██████████| 500/500 [00:03<00:00, 134.98it/s]


  AMST                : MAP = 51.65%


Retrieval: 100%|██████████| 500/500 [00:10<00:00, 47.17it/s]

  AMST+ViT+HOG        : MAP = 67.29%


## Cell 24 — Retrieval Evaluation: Bullseye Score (Standard MPEG-7 Protocol)

In addition to Mean Average Precision, retrieval performance is reported using the Bullseye score, the standard evaluation protocol for MPEG-7 CE-Shape-1 in the shape-retrieval literature. Each image is used as a query against every other image in the dataset; the number of same-class images appearing among the top-40 nearest neighbours is divided by the class size (20) and averaged over the full dataset. Reporting this metric allows direct, protocol-matched comparison with Shape Context, IDSC, and related prior work.


In [25]:
def bullseye_score(X, y, metric='cosine', top_k=40, class_size=20):
    """Standard MPEG-7 CE-Shape-1 Bullseye evaluation: every image is queried against every
    other image; the number of same-class images among the top-40 nearest neighbours is
    divided by the class size and averaged over the dataset."""
    X = np.nan_to_num(X.copy())
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    y = np.asarray(y)
    if metric == 'cosine':
        norms = np.linalg.norm(Xs, axis=1, keepdims=True) + 1e-12
        Xn = Xs / norms
        sim = Xn @ Xn.T
        np.fill_diagonal(sim, -np.inf)
        order = np.argsort(-sim, axis=1)
    else:
        from scipy.spatial.distance import cdist
        dist = cdist(Xs, Xs)
        np.fill_diagonal(dist, np.inf)
        order = np.argsort(dist, axis=1)
    top = order[:, :top_k]
    hits = (y[top] == y[:, None]).sum(axis=1)
    return float(np.mean(hits / class_size)) * 100

print('Bullseye Score evaluation (MPEG-7 CE-Shape-1 standard protocol, top-40 / class size 20):')
bullseye_results = {}
for name, X_ret in [('HOG', X_hog), ('ViT-B/16', X_vit), ('AMST', X_amst), ('AMST+ViT+HOG', X_comb_mpeg)]:
    bs = bullseye_score(X_ret, y_mpeg)
    bullseye_results[name] = bs
    print(f'  {name:20s}: Bullseye = {bs:.2f}%')

pd.DataFrame([{'Method': k, 'Bullseye': f'{v:.2f}%'} for k, v in bullseye_results.items()]).to_csv(
    'bullseye_results.csv', index=False)
print('\\nSaved to bullseye_results.csv')


Bullseye Score evaluation (MPEG-7 CE-Shape-1 standard protocol, top-40 / class size 20):
  HOG                 : Bullseye = 56.82%
  ViT-B/16            : Bullseye = 66.65%
  AMST                : Bullseye = 56.03%
  AMST+ViT+HOG        : Bullseye = 68.76%
\nSaved to bullseye_results.csv


## Figure 4 — Precision-Recall Curves


In [26]:
def pr_curve_rank_fusion(X, y, comp_dims, metric='cosine'):
    X_raw = np.nan_to_num(X.copy())
    X_n = np.zeros_like(X_raw)
    start = 0
    for dim in comp_dims:
        end = start+dim
        mu = X_raw[:, start:end].mean(axis=0)
        std = X_raw[:, start:end].std(axis=0)+1e-10
        X_n[:, start:end] = (X_raw[:, start:end]-mu)/std
        start = end
    N = len(y)
    n_q = min(N, 300)
    q_idx = np.random.RandomState(SEED).choice(N, n_q, replace=False)
    all_P, all_R = [], []
    for qi in tqdm(q_idx, desc='PR curve'):
        all_ranks = np.zeros((len(comp_dims), N))
        start = 0
        for ci, dim in enumerate(comp_dims):
            end = start+dim
            Xc = X_n[:, start:end]
            if metric == 'cosine':
                sim = (Xc @ Xc[qi]) / (np.linalg.norm(Xc, axis=1) * np.linalg.norm(Xc[qi]) + 1e-12)
                all_ranks[ci] = np.argsort(np.argsort(-sim))
            else:
                dist = np.sqrt(((Xc - Xc[qi])**2).sum(axis=1))
                all_ranks[ci] = np.argsort(np.argsort(dist))
            start = end
        fused_rank = all_ranks.mean(axis=0)
        ranked = np.argsort(fused_rank)
        ranked = ranked[ranked != qi]
        rel = (y[ranked] == y[qi]).astype(int)
        if rel.sum()==0: continue
        cs = np.cumsum(rel)
        all_P.append(cs/np.arange(1,len(ranked)+1))
        all_R.append(cs/rel.sum())
    rc = np.linspace(0,1,20)
    ip = [np.interp(rc, r, p) for p,r in zip(all_P, all_R)]
    return rc, np.mean(ip, axis=0)

def pr_curve_single(X, y, metric='cosine'):
    X_use = StandardScaler().fit_transform(np.nan_to_num(X))
    N = len(y)
    n_q = min(N, 300)
    q_idx = np.random.RandomState(SEED).choice(N, n_q, replace=False)
    all_P, all_R = [], []
    for qi in tqdm(q_idx, desc='PR curve'):
        if metric == 'cosine':
            sim = (X_use @ X_use[qi]) / (np.linalg.norm(X_use, axis=1) * np.linalg.norm(X_use[qi]) + 1e-12)
            ranked = np.argsort(-sim)[1:]
        else:
            dist = np.sqrt(((X_use - X_use[qi])**2).sum(axis=1))
            ranked = np.argsort(dist)[1:]
        rel = (y[ranked] == y[qi]).astype(int)
        if rel.sum()==0: continue
        cs = np.cumsum(rel)
        all_P.append(cs/np.arange(1,len(ranked)+1))
        all_R.append(cs/rel.sum())
    rc = np.linspace(0,1,20)
    ip = [np.interp(rc, r, p) for p,r in zip(all_P, all_R)]
    return rc, np.mean(ip, axis=0)

pr_configs = [
    ('HOG', X_hog),
    ('ViT-B/16', X_vit),
    ('AMST', X_amst),
    ('AMST+ViT+HOG', X_comb_mpeg),
]
pr_line_styles = ['--', '-.', ':', '-']
pr_colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c']
fig, ax = plt.subplots(figsize=(11, 8))
for idx, (name, X_p) in enumerate(pr_configs):
    rc, prec = pr_curve_single(X_p, y_mpeg, 'cosine')
    map_val = retrieval_results.get(name, 0)*100
    auc_pr = np.trapz(prec, rc)
    ax.plot(rc, prec, lw=2.5, linestyle=pr_line_styles[idx], color=pr_colors[idx],
            label=f'{name} (MAP={map_val:.1f}%, AUC={auc_pr:.2f})')
    ax.fill_between(rc, prec, alpha=0.08, color=pr_colors[idx])
ax.set_xlabel('Recall', fontsize=14, fontweight='bold')
ax.set_ylabel('Precision', fontsize=14, fontweight='bold')
ax.set_title('Precision-Recall Curves — MPEG-7 Shape Retrieval', fontsize=15, fontweight='bold')
ax.legend(loc='best', fontsize=11, frameon=True, fancybox=True)
ax.grid(alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
sns.despine()
plt.tight_layout()
plt.savefig('fig4_precision_recall.png', dpi=200, bbox_inches='tight')
plt.show()
print('Figure 4 saved.')


PR curve: 100%|██████████| 300/300 [00:05<00:00, 50.99it/s]


Figure 4 saved.


## Cell 25 — Cross-Dataset Generalization via Kimia-216 Retrieval


In [27]:
mpeg_classes = le.classes_
kimia_classes = le_kimia.classes_
mpeg_set = set(mpeg_classes)
kimia_set = set(kimia_classes)
common = mpeg_set & kimia_set
print(f'MPEG-7 classes: {len(mpeg_set)}, Kimia classes: {len(kimia_set)}, Common: {len(common)}')

kimia_common_idx = [i for i, lbl in enumerate(kim_labels) if lbl in common]
print(f'Kimia queries with common labels: {len(kimia_common_idx)}/{len(kim_labels)}')

kimia_to_mpeg_y = []
for i in kimia_common_idx:
    lbl = kim_labels[i]
    mpeg_idx = np.where(mpeg_classes == lbl)[0][0]
    kimia_to_mpeg_y.append(mpeg_idx)

def cross_domain_single(gallery_X, gallery_y, query_X, query_y, metric='cosine'):
    scaler = StandardScaler()
    Xg_n = scaler.fit_transform(np.nan_to_num(gallery_X))
    Xq_n = scaler.transform(np.nan_to_num(query_X))
    APs = []
    for qi in range(len(query_y)):
        if metric == 'cosine':
            sim = (Xg_n @ Xq_n[qi]) / (np.linalg.norm(Xg_n, axis=1)*np.linalg.norm(Xq_n[qi])+1e-12)
            ranked = np.argsort(-sim)
        else:
            dist = np.sqrt(((Xg_n - Xq_n[qi])**2).sum(axis=1))
            ranked = np.argsort(dist)
        rel = (gallery_y[ranked]==query_y[qi]).astype(int)
        if rel.sum()==0: continue
        cs = np.cumsum(rel)
        pos = np.arange(1, len(ranked)+1)
        APs.append((cs/pos * rel).sum() / rel.sum())
    return float(np.mean(APs)) if APs else 0.0

def cross_domain_rank_fusion(gallery_X, gallery_y, query_X, query_y, comp_dims, metric='cosine'):
    Xg_n = np.zeros_like(gallery_X)
    Xq_n = np.zeros_like(query_X)
    start = 0
    for dim in comp_dims:
        end = start+dim
        mu = gallery_X[:, start:end].mean(axis=0)
        std = gallery_X[:, start:end].std(axis=0)+1e-10
        Xg_n[:, start:end] = (gallery_X[:, start:end]-mu)/std
        Xq_n[:, start:end] = (query_X[:, start:end]-mu)/std
        start = end
    APs = []
    Nq = len(query_y)
    for qi in tqdm(range(Nq), desc='Cross rank fusion'):
        all_ranks = np.zeros((len(comp_dims), len(gallery_y)))
        start = 0
        for ci, dim in enumerate(comp_dims):
            end = start+dim
            Xgc = Xg_n[:, start:end]
            Xqc = Xq_n[qi, start:end]
            if metric == 'cosine':
                sim = (Xgc @ Xqc) / (np.linalg.norm(Xgc, axis=1) * np.linalg.norm(Xqc) + 1e-12)
                all_ranks[ci] = np.argsort(np.argsort(-sim))
            else:
                dist = np.sqrt(((Xgc - Xqc)**2).sum(axis=1))
                all_ranks[ci] = np.argsort(np.argsort(dist))
            start = end
        fused = all_ranks.mean(axis=0)
        ranked = np.argsort(fused)
        rel = (gallery_y[ranked]==query_y[qi]).astype(int)
        if rel.sum()==0: continue
        cs = np.cumsum(rel)
        pos = np.arange(1, len(ranked)+1)
        APs.append((cs/pos * rel).sum() / rel.sum())
    return float(np.mean(APs)) if APs else 0.0

cross_results = {}
kimia_common_idx = [i for i, lbl in enumerate(kim_labels) if lbl in common]
print(f'Kimia queries with common labels: {len(kimia_common_idx)}/{len(kim_labels)}')

if len(kimia_common_idx) >= 5:
    kimia_to_mpeg_y = []
    for i in kimia_common_idx:
        lbl = kim_labels[i]
        mpeg_idx = np.where(mpeg_classes == lbl)[0][0]
        kimia_to_mpeg_y.append(mpeg_idx)
    yq_aligned = np.array(kimia_to_mpeg_y)
    print('\nCross-dataset retrieval MAP (gallery=MPEG-7, query=Kimia-216, global cosine):')
    for name, Xg, Xq in [('HOG', X_hog, X_kim_hog), ('ViT-B/16', X_vit, X_kim_vit),
                           ('AMST', X_amst, X_kim_amst), ('AMST+ViT+HOG', X_comb_mpeg, X_comb_kimia)]:
        Xq_sub = Xq[kimia_common_idx]
        map_val = cross_domain_single(Xg, y_mpeg, Xq_sub, yq_aligned, 'cosine')
        cross_results[name] = map_val
        print(f'  {name:25s}: MAP = {map_val*100:.2f}%')
else:
    print(f'Too few common classes ({len(kimia_common_idx)}), using all Kimia queries...')
    for name, Xg, Xq in [('HOG', X_hog, X_kim_hog), ('ViT-B/16', X_vit, X_kim_vit),
                           ('AMST', X_amst, X_kim_amst), ('AMST+ViT+HOG', X_comb_mpeg, X_comb_kimia)]:
        map_val = cross_domain_single(Xg, y_mpeg, Xq, y_kimia, 'cosine')
        cross_results[name] = map_val
        print(f'  {name:25s}: MAP = {map_val*100:.2f}%')


MPEG-7 classes: 70, Kimia classes: 18, Common: 18
Kimia queries with common labels: 216/216
Kimia queries with common labels: 216/216

Cross-dataset retrieval MAP (gallery=MPEG-7, query=Kimia-216, global cosine):
  HOG                      : MAP = 34.50%
  ViT-B/16                 : MAP = 16.07%
  AMST                     : MAP = 6.43%
  AMST+ViT+HOG             : MAP = 16.71%


## Cell 26 — Cross-Dataset Component-wise Diagnostic

To understand which part of the AMST descriptor is responsible for its cross-dataset MAP drop (51.65% within MPEG-7 versus 6.43% on the MPEG-7 -> Kimia-216 transfer task), this cell repeats the cross-domain retrieval evaluation using the same cumulative component subsets as the within-dataset ablation study (Cell 18), and additionally reports the per-class mean Average Precision for the full AMST descriptor so that the classes driving the drop can be identified directly rather than attributed only in general terms.


In [28]:
# --- Component-wise cross-dataset MAP (mirrors the within-dataset ablation configs) ---
cross_ablation_results = {}
for name, comp_dims in ablation_configs.items():
    total_d = sum(comp_dims)
    start = 0
    X_sub_mpeg = np.zeros((len(all_images), total_d))
    for dim in comp_dims:
        end = start + dim
        X_sub_mpeg[:, start:end] = X_amst[:, start:end]
        start = end
    start = 0
    X_sub_kim = np.zeros((len(kim_images), total_d))
    for dim in comp_dims:
        end = start + dim
        X_sub_kim[:, start:end] = X_kim_amst[:, start:end]
        start = end
    X_sub_kim_q = X_sub_kim[kimia_common_idx]
    map_val = cross_domain_single(X_sub_mpeg, y_mpeg, X_sub_kim_q, yq_aligned, 'cosine')
    cross_ablation_results[name] = map_val * 100
    print(f'  {name:30s}: Cross-MAP = {map_val*100:6.2f}%')

pd.DataFrame([{'Config': k, 'Cross_MAP': f'{v:.2f}%'} for k, v in cross_ablation_results.items()]
             ).to_csv('cross_dataset_ablation.csv', index=False)
print('Saved to cross_dataset_ablation.csv')

# --- Per-class cross-dataset AP for the full AMST descriptor ---
def cross_domain_single_per_query(gallery_X, gallery_y, query_X, query_y, metric='cosine'):
    scaler = StandardScaler()
    Xg_n = scaler.fit_transform(np.nan_to_num(gallery_X))
    Xq_n = scaler.transform(np.nan_to_num(query_X))
    APs = []
    for qi in range(len(query_y)):
        if metric == 'cosine':
            sim = (Xg_n @ Xq_n[qi]) / (np.linalg.norm(Xg_n, axis=1) * np.linalg.norm(Xq_n[qi]) + 1e-12)
            ranked = np.argsort(-sim)
        else:
            dist = np.sqrt(((Xg_n - Xq_n[qi]) ** 2).sum(axis=1))
            ranked = np.argsort(dist)
        rel = (gallery_y[ranked] == query_y[qi]).astype(int)
        if rel.sum() == 0:
            APs.append(0.0)
            continue
        cs = np.cumsum(rel)
        pos = np.arange(1, len(ranked) + 1)
        APs.append((cs / pos * rel).sum() / rel.sum())
    return np.array(APs)

ap_per_query = cross_domain_single_per_query(X_amst, y_mpeg, X_kim_amst[kimia_common_idx], yq_aligned)
per_class_records = []
for cls_idx in np.unique(yq_aligned):
    mask = yq_aligned == cls_idx
    per_class_records.append({
        'Class': mpeg_classes[cls_idx],
        'Mean_AP_pct': round(float(ap_per_query[mask].mean()) * 100, 2),
        'N_Queries': int(mask.sum()),
    })
per_class_df = pd.DataFrame(per_class_records).sort_values('Mean_AP_pct')
per_class_df.to_csv('cross_dataset_per_class_amst.csv', index=False)
print('\\nWorst-transferring classes (AMST, MPEG-7 -> Kimia-216):')
print(per_class_df.head(10).to_string(index=False))
print('\\nBest-transferring classes (AMST, MPEG-7 -> Kimia-216):')
print(per_class_df.tail(10).to_string(index=False))
print('\\nFull per-class breakdown saved to cross_dataset_per_class_amst.csv')


  C1: APCFW+                    : Cross-MAP =  11.44%
  C1+C3: +SPD                   : Cross-MAP =   9.53%
  C1-C3: +Topology              : Cross-MAP =   8.22%
  C1-C4: +Morphology            : Cross-MAP =   5.97%
  Full AMST                     : Cross-MAP =   6.43%
Saved to cross_dataset_ablation.csv
\nWorst-transferring classes (AMST, MPEG-7 -> Kimia-216):
   Class  Mean_AP_pct  N_Queries
  hammer         0.96         12
    fork         1.07         12
    bone         1.54         12
children         1.64         12
    bird         1.67         12
     ray         1.69         12
  turtle         1.91         12
    misk         1.92         12
     car         1.97         12
   heart         2.25         12
\nBest-transferring classes (AMST, MPEG-7 -> Kimia-216):
   Class  Mean_AP_pct  N_Queries
     car         1.97         12
   heart         2.25         12
 classic         2.49         12
     key         3.13         12
   brick         3.95         12
   camel         4

## Figure 5 — Cross-Dataset Retrieval Comparison


In [29]:
if cross_results:
    fig, ax = plt.subplots(figsize=(10, 6))
    names = list(cross_results.keys())
    maps = [cross_results[n]*100 for n in names]
    colors_cross = ['#e74c3c' if n == 'AMST+ViT+HOG' else ('#2ecc71' if n == 'HOG' else '#3498db') for n in names]
    bars = ax.bar(names, maps, color=colors_cross, edgecolor='black', linewidth=1.2, width=0.55)
    for bar, v in zip(bars, maps):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4, f'{v:.1f}%',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_ylabel('Retrieval MAP (%)', fontsize=14, fontweight='bold')
    ax.set_title('Cross-Dataset Generalization (MPEG-7 → Kimia-216)', fontsize=14, fontweight='bold')
    ax.set_ylim(0, max(45, max(maps)*1.25))
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, fontsize=11)
    ax.axhline(y=16.71, color='#e74c3c', linestyle='--', linewidth=1.2, alpha=0.5, label='AMST+ViT+HOG: 16.71%')
    ax.legend(fontsize=10)
    plt.grid(axis='y', alpha=0.3)
    sns.despine()
    plt.tight_layout()
    plt.savefig('fig5_cross_dataset.png', dpi=200, bbox_inches='tight')
    plt.show()
    print('Figure 5 saved.')


Figure 5 saved.


## Cell 29 — Synthetic Controlled Invariance Benchmark: Dataset Generation

MPEG-7 CE-Shape-1 and Kimia-216 are both real-world silhouette collections, so neither allows individual geometric factors (rotation, scale, local deformation) to be varied independently while holding shape identity fixed. This section introduces a small, fully synthetic, procedurally generated shape benchmark (Synthetic Geometric Shape Benchmark, SGSB) built from ten parametric polygon families (triangle, square, pentagon, hexagon, octagon, two star variants, a cross, an ellipse, and an arrow). SGSB serves two purposes: (i) an independent classification benchmark on a shape distribution disjoint from MPEG-7, and (ii) a controlled testbed in which rotation, scale, and elastic deformation can be applied one at a time to directly test the invariance properties claimed for the AMST descriptor (Cell 7). All generation is deterministic given SEED=42 and uses only NumPy/PIL primitives, so it requires no external download and runs quickly on CPU.


In [30]:
import math
from PIL import Image as PILImage, ImageDraw

def _regular_polygon(n_sides, rotation=0.0):
    angles = np.linspace(0, 2*np.pi, n_sides, endpoint=False) + rotation
    return np.column_stack([np.cos(angles), np.sin(angles)])

def _star_polygon(n_points, inner_ratio=0.5, rotation=0.0):
    angles = np.linspace(0, 2*np.pi, n_points*2, endpoint=False) + rotation
    radii = np.array([1.0 if i % 2 == 0 else inner_ratio for i in range(n_points*2)])
    return np.column_stack([radii*np.cos(angles), radii*np.sin(angles)])

def _cross_polygon(arm_ratio=0.32):
    a = arm_ratio
    pts = [(-a,1),(a,1),(a,a),(1,a),(1,-a),(a,-a),(a,-1),(-a,-1),(-a,-a),(-1,-a),(-1,a),(-a,a)]
    return np.array(pts, dtype=float)

def _ellipse_polygon(aspect=0.6, n=48):
    t = np.linspace(0, 2*np.pi, n, endpoint=False)
    return np.column_stack([np.cos(t), aspect*np.sin(t)])

def _arrow_polygon():
    pts = [(-1,0.28),(0.15,0.28),(0.15,0.62),(1,0),(0.15,-0.62),(0.15,-0.28),(-1,-0.28)]
    return np.array(pts, dtype=float)

SGSB_CLASSES = {
    'triangle': lambda: _regular_polygon(3),
    'square':   lambda: _regular_polygon(4, rotation=np.pi/4),
    'pentagon': lambda: _regular_polygon(5),
    'hexagon':  lambda: _regular_polygon(6),
    'octagon':  lambda: _regular_polygon(8),
    'star5':    lambda: _star_polygon(5),
    'star6':    lambda: _star_polygon(6, inner_ratio=0.6),
    'cross':    lambda: _cross_polygon(),
    'ellipse':  lambda: _ellipse_polygon(),
    'arrow':    lambda: _arrow_polygon(),
}

def rasterize_polygon(base_pts, size=IMG_SIZE, rotation=0.0, scale=1.0, shear=0.0,
                       jitter=0.0, rng=None):
    """Apply rotation/scale/shear/elastic-jitter to a base polygon and rasterize to a
    binary silhouette image using the same conventions (bool mask, uint8) as load_binarize."""
    rng = rng if rng is not None else np.random.RandomState(0)
    p = base_pts.copy()
    if jitter > 0:
        p = p + rng.normal(0, jitter, size=p.shape)
    c, s = np.cos(rotation), np.sin(rotation)
    R = np.array([[c, -s], [s, c]])
    p = p @ R.T
    Sh = np.array([[1.0, shear], [0.0, 1.0]])
    p = p @ Sh.T
    p = p * scale
    margin = 0.15
    half = (size[0] / 2) * (1 - margin)
    px = p[:, 0]*half + size[0]/2
    py = p[:, 1]*half + size[1]/2
    img = PILImage.new('L', size, 0)
    draw = ImageDraw.Draw(img)
    draw.polygon(list(zip(px.tolist(), py.tolist())), fill=255)
    bw = (np.array(img) > 127).astype(np.uint8)
    bw = closing(bw, disk(1))
    bw = opening(bw, disk(1))
    return bw

SGSB_CACHE = 'sgsb_dataset.npz'
SGSB_N_PER_CLASS = 20

if Path(SGSB_CACHE).exists():
    _c = np.load(SGSB_CACHE, allow_pickle=True)
    sgsb_images, sgsb_contours, sgsb_labels = _c['images'], _c['contours'], list(_c['labels'])
    print(f'Loaded cached SGSB: {len(sgsb_images)} images')
else:
    sgsb_images, sgsb_labels = [], []
    rng_sgsb = np.random.RandomState(SEED + 7)
    for cls_name, gen_fn in SGSB_CLASSES.items():
        base_pts = gen_fn()
        for k in range(SGSB_N_PER_CLASS):
            rot = rng_sgsb.uniform(0, 2*np.pi)
            scl = rng_sgsb.uniform(0.85, 1.15)
            shr = rng_sgsb.uniform(-0.08, 0.08)
            jit = rng_sgsb.uniform(0.0, 0.02)
            bw = rasterize_polygon(base_pts, rotation=rot, scale=scl, shear=shr, jitter=jit, rng=rng_sgsb)
            sgsb_images.append(bw)
            sgsb_labels.append(cls_name)
    sgsb_images = np.array(sgsb_images, dtype=np.uint8)
    sgsb_contours = np.array([get_contour(bw) for bw in sgsb_images], dtype=np.float32)
    np.savez_compressed(SGSB_CACHE, images=sgsb_images, contours=sgsb_contours,
                         labels=np.array(sgsb_labels))

le_sgsb = LabelEncoder()
y_sgsb = le_sgsb.fit_transform(sgsb_labels)
print(f'Synthetic Geometric Shape Benchmark (SGSB): {len(sgsb_images)} images, '
      f'{len(le_sgsb.classes_)} classes, {SGSB_N_PER_CLASS} instances/class')

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
class_to_first = {}
for i, lbl in enumerate(sgsb_labels):
    if lbl not in class_to_first:
        class_to_first[lbl] = i
for ax, (cls, idx) in zip(axes.flat, class_to_first.items()):
    ax.imshow(sgsb_images[idx], cmap='gray_r')
    ax.set_title(cls, fontsize=11)
    ax.axis('off')
plt.suptitle('SGSB — One Example per Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig10_sgsb_samples.png', dpi=200, bbox_inches='tight')
plt.show()
print('Figure 10 saved.')


Synthetic Geometric Shape Benchmark (SGSB): 200 images, 10 classes, 20 instances/class
Figure 10 saved.


## Cell 30 — SGSB Feature Extraction and Classification

AMST, HOG, and ViT-B/16 features are extracted for every SGSB image using the identical descriptor functions defined earlier in this notebook, and the combined AMST+ViT+HOG representation is built the same way as for MPEG-7. A 5-fold stratified classification (reduced from 10-fold because of the smaller dataset size) reports whether the relative ranking of methods observed on MPEG-7 holds on a shape distribution the descriptors were not designed around.


In [31]:
SGSB_DL_CACHE = 'sgsb_dl_features.npz'
if Path(SGSB_DL_CACHE).exists():
    _d = np.load(SGSB_DL_CACHE)
    X_sgsb_vit = _d['X_vit']
    print(f'Loaded cached SGSB ViT features: {X_sgsb_vit.shape}')
else:
    _vit_sgsb = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0).to(DEVICE).eval()
    _tfms_sgsb = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    def _vit_feat_sgsb(bw):
        rgb = np.stack([bw]*3, axis=-1).astype(np.float32)
        img = transforms.ToPILImage()(rgb)
        x = _tfms_sgsb(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            return _vit_sgsb(x).cpu().numpy().flatten()
    X_sgsb_vit = np.array([_vit_feat_sgsb(bw) for bw in tqdm(sgsb_images, desc='SGSB ViT')])
    np.savez_compressed(SGSB_DL_CACHE, X_vit=X_sgsb_vit)
    del _vit_sgsb

X_sgsb_amst = np.array([amst_descriptor(bw, c) for bw, c in
                         tqdm(zip(sgsb_images, sgsb_contours), total=len(sgsb_images), desc='SGSB AMST')])
X_sgsb_hog = np.array([hog_descriptor(bw) for bw in sgsb_images])
X_sgsb_comb = build_combined(X_sgsb_amst, X_sgsb_vit, X_sgsb_hog)
print(f'SGSB features: AMST{X_sgsb_amst.shape}, HOG{X_sgsb_hog.shape}, '
      f'ViT{X_sgsb_vit.shape}, Combined{X_sgsb_comb.shape}')

def eval_generic(X_raw, y, n_folds, comp_dims=None, k_features=0):
    X_raw = np.nan_to_num(X_raw.copy())
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    accs = []
    for tr, te in skf.split(X_raw, y):
        if comp_dims is not None:
            Xtr, Xte = per_component_normalize(X_raw, comp_dims, tr, te)
        else:
            sc = StandardScaler()
            Xtr = sc.fit_transform(X_raw[tr]); Xte = sc.transform(X_raw[te])
        if k_features > 0:
            k = min(k_features, Xtr.shape[1])
            sel = SelectKBest(f_classif, k=k)
            Xtr = sel.fit_transform(Xtr, y[tr]); Xte = sel.transform(Xte)
        if comp_dims is not None:
            sc2 = StandardScaler()
            Xtr = sc2.fit_transform(Xtr); Xte = sc2.transform(Xte)
        clf = SVC(kernel='rbf', C=100, gamma='scale', decision_function_shape='ovr', random_state=SEED)
        clf.fit(Xtr, y[tr])
        accs.append(accuracy_score(y[te], clf.predict(Xte)))
    return np.array(accs) * 100

N_FOLDS_SGSB = 5
sgsb_results = {}
sgsb_results['HOG'] = eval_generic(X_sgsb_hog, y_sgsb, N_FOLDS_SGSB)
sgsb_results['AMST'] = eval_generic(X_sgsb_amst, y_sgsb, N_FOLDS_SGSB,
                                     comp_dims=[160, 90, 210, 128, 30], k_features=350)
sgsb_results['ViT-B/16'] = eval_generic(X_sgsb_vit, y_sgsb, N_FOLDS_SGSB)
sgsb_results['AMST+ViT+HOG'] = eval_generic(X_sgsb_comb, y_sgsb, N_FOLDS_SGSB,
                                             comp_dims=COMBINED_COMP_DIMS, k_features=1000)

print(f'\\nSGSB classification ({N_FOLDS_SGSB}-fold stratified CV):')
for nm in sorted(sgsb_results, key=lambda n: sgsb_results[n].mean(), reverse=True):
    print(f'  {nm:20s}: {sgsb_results[nm].mean():6.2f}% +/-{sgsb_results[nm].std():4.2f}%')

pd.DataFrame([{'Method': nm, 'Accuracy': f'{v.mean():.2f}%', 'Std': f'{v.std():.2f}%'}
              for nm, v in sgsb_results.items()]).to_csv('sgsb_classification_results.csv', index=False)
print('Saved to sgsb_classification_results.csv')


SGSB AMST: 100%|██████████| 200/200 [01:27<00:00,  2.28it/s]


SGSB features: AMST(200, 618), HOG(200, 324), ViT(200, 768), Combined(200, 1710)
\nSGSB classification (5-fold stratified CV):
  ViT-B/16            : 100.00% +/-0.00%
  AMST+ViT+HOG        :  99.50% +/-1.00%
  AMST                :  98.00% +/-2.45%
  HOG                 :  83.00% +/-4.30%
Saved to sgsb_classification_results.csv


## Cell 31 — Controlled Invariance Diagnostic (Rotation / Scale / Elastic Deformation)

This cell isolates the three geometric factors that are conflated in any real-world cross-dataset comparison. Classifiers for all four methods are trained once on a canonical SGSB training set (near-identity rotation, scale, and shear). Three disjoint test suites are then generated, each varying exactly one factor — pure rotation, pure scale, or pure elastic vertex jitter — while holding the other two at their canonical values, at increasing magnitude. Plotting accuracy against magnitude for each factor separately shows directly which invariance properties the AMST descriptor (and its C1/APCFW+ component in particular) does and does not hold in practice, providing a controlled, causal complement to the aggregate MPEG-7 -> Kimia-216 result in Cell 25.


In [32]:
N_INV_LEVELS = 6
N_INV_REPEATS = 3
N_INV_PER_LEVEL = 50   # 5 images x 10 classes

rotation_levels = np.linspace(0, np.pi, N_INV_LEVELS)                 # 0 to 180 degrees
scale_levels = np.linspace(0.0, 0.30, N_INV_LEVELS)                    # shrink factor 0-30%
deform_levels = np.linspace(0.0, 0.12, N_INV_LEVELS)                   # elastic jitter magnitude

# --- canonical training set (near-identity transforms only) ---
rng_train = np.random.RandomState(SEED + 11)
train_images, train_labels = [], []
for cls_name, gen_fn in SGSB_CLASSES.items():
    base_pts = gen_fn()
    for k in range(20):
        jit = rng_train.uniform(0.0, 0.01)
        bw = rasterize_polygon(base_pts, rotation=0.0, scale=1.0, shear=0.0, jitter=jit, rng=rng_train)
        train_images.append(bw)
        train_labels.append(cls_name)
train_images = np.array(train_images, dtype=np.uint8)
y_inv_train = le_sgsb.transform(train_labels)
train_contours = np.array([get_contour(bw) for bw in train_images], dtype=np.float32)

X_it_amst = np.array([amst_descriptor(bw, c) for bw, c in zip(train_images, train_contours)])
X_it_hog = np.array([hog_descriptor(bw) for bw in train_images])

_vit_inv = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0).to(DEVICE).eval()
_tfms_inv = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
def _vit_feat_inv(bw):
    rgb = np.stack([bw]*3, axis=-1).astype(np.float32)
    img = transforms.ToPILImage()(rgb)
    x = _tfms_inv(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        return _vit_inv(x).cpu().numpy().flatten()

X_it_vit = np.array([_vit_feat_inv(bw) for bw in tqdm(train_images, desc='Invariance train ViT')])
X_it_comb = build_combined(X_it_amst, X_it_vit, X_it_hog)

# --- fit one classifier per method on the canonical training set ---
sc_it_h = StandardScaler(); Xh = sc_it_h.fit_transform(np.nan_to_num(X_it_hog))
clf_it_h = SVC(kernel='rbf', C=100, gamma='scale', decision_function_shape='ovr', random_state=SEED)
clf_it_h.fit(Xh, y_inv_train)

COMP_DIMS_LIST5 = [160, 90, 210, 128, 30]
Xa_n = np.zeros_like(X_it_amst); start = 0
for dim in COMP_DIMS_LIST5:
    end = start + dim
    mu = X_it_amst[:, start:end].mean(axis=0); std = X_it_amst[:, start:end].std(axis=0) + 1e-10
    Xa_n[:, start:end] = (X_it_amst[:, start:end] - mu) / std
    start = end
sel_it_a = SelectKBest(f_classif, k=min(350, Xa_n.shape[1]))
Xa_s = sel_it_a.fit_transform(Xa_n, y_inv_train)
sc_it_a = StandardScaler(); Xa_f = sc_it_a.fit_transform(Xa_s)
clf_it_a = SVC(kernel='rbf', C=100, gamma='scale', decision_function_shape='ovr', random_state=SEED)
clf_it_a.fit(Xa_f, y_inv_train)

sc_it_v = StandardScaler(); Xv = sc_it_v.fit_transform(np.nan_to_num(X_it_vit))
clf_it_v = SVC(kernel='rbf', C=100, gamma='scale', decision_function_shape='ovr', random_state=SEED)
clf_it_v.fit(Xv, y_inv_train)

sel_it_c = SelectKBest(f_classif, k=min(1000, X_it_comb.shape[1]))
Xc_n = np.zeros_like(X_it_comb); start = 0
for dim in COMBINED_COMP_DIMS:
    end = start + dim
    mu = X_it_comb[:, start:end].mean(axis=0); std = X_it_comb[:, start:end].std(axis=0) + 1e-10
    Xc_n[:, start:end] = (X_it_comb[:, start:end] - mu) / std
    start = end
Xc_s = sel_it_c.fit_transform(Xc_n, y_inv_train)
sc_it_c = StandardScaler(); Xc_f = sc_it_c.fit_transform(Xc_s)
clf_it_c = SVC(kernel='rbf', C=100, gamma='scale', decision_function_shape='ovr', random_state=SEED)
clf_it_c.fit(Xc_f, y_inv_train)


def _generate_level_batch(transform_type, magnitude, rng):
    imgs, labels = [], []
    per_class = max(1, N_INV_PER_LEVEL // len(SGSB_CLASSES))
    for cls_name, gen_fn in SGSB_CLASSES.items():
        base_pts = gen_fn()
        for _ in range(per_class):
            rot = magnitude if transform_type == 'rotation' else 0.0
            scl = (1.0 - magnitude) if transform_type == 'scale' else 1.0
            jit = magnitude if transform_type == 'deform' else 0.01
            bw = rasterize_polygon(base_pts, rotation=rot, scale=scl, shear=0.0, jitter=jit, rng=rng)
            imgs.append(bw)
            labels.append(cls_name)
    return imgs, le_sgsb.transform(labels)


def _score_batch(imgs, y_batch):
    contours = [get_contour(bw) for bw in imgs]
    hog_f = np.nan_to_num(np.array([hog_descriptor(bw) for bw in imgs]))
    amst_f = np.nan_to_num(np.array([amst_descriptor(bw, c) for bw, c in zip(imgs, contours)]))
    vit_f = np.nan_to_num(np.array([_vit_feat_inv(bw) for bw in imgs]))
    comb_f = np.concatenate([amst_f, vit_f, hog_f], axis=1)

    acc_h = clf_it_h.score(sc_it_h.transform(hog_f), y_batch) * 100

    an = np.zeros_like(amst_f); start = 0
    for dim in COMP_DIMS_LIST5:
        end = start + dim
        mu = amst_f[:, start:end].mean(axis=0); std = amst_f[:, start:end].std(axis=0) + 1e-10
        an[:, start:end] = (amst_f[:, start:end] - mu) / std
        start = end
    acc_a = clf_it_a.score(sc_it_a.transform(sel_it_a.transform(an)), y_batch) * 100

    acc_v = clf_it_v.score(sc_it_v.transform(vit_f), y_batch) * 100

    cn = np.zeros_like(comb_f); start = 0
    for dim in COMBINED_COMP_DIMS:
        end = start + dim
        mu = comb_f[:, start:end].mean(axis=0); std = comb_f[:, start:end].std(axis=0) + 1e-10
        cn[:, start:end] = (comb_f[:, start:end] - mu) / std
        start = end
    acc_c = clf_it_c.score(sc_it_c.transform(sel_it_c.transform(cn)), y_batch) * 100

    return {'HOG': acc_h, 'AMST': acc_a, 'ViT-B/16': acc_v, 'AMST+ViT+HOG': acc_c}


invariance_results = {}
for transform_type, levels in [('rotation', rotation_levels), ('scale', scale_levels), ('deform', deform_levels)]:
    per_level = {m: [] for m in ['HOG', 'AMST', 'ViT-B/16', 'AMST+ViT+HOG']}
    for lvl in tqdm(levels, desc=f'Invariance: {transform_type}'):
        level_scores = {m: [] for m in per_level}
        for rep in range(N_INV_REPEATS):
            rng = np.random.RandomState(SEED * 3000 + rep)
            imgs, y_batch = _generate_level_batch(transform_type, lvl, rng)
            scores = _score_batch(imgs, y_batch)
            for m in level_scores:
                level_scores[m].append(scores[m])
        for m in per_level:
            per_level[m].append(level_scores[m])
    invariance_results[transform_type] = per_level

del _vit_inv
if torch.cuda.is_available():
    torch.cuda.empty_cache()

inv_records = []
for transform_type, levels in [('rotation', rotation_levels), ('scale', scale_levels), ('deform', deform_levels)]:
    for method, per_level in invariance_results[transform_type].items():
        for lvl, scores in zip(levels, per_level):
            for s in scores:
                inv_records.append({'Transform': transform_type, 'Magnitude': float(lvl),
                                     'Method': method, 'Accuracy': s})
pd.DataFrame(inv_records).to_csv('invariance_diagnostic_raw.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(19, 6))
colors_inv = {'HOG': '#3498db', 'AMST': '#e67e22', 'ViT-B/16': '#2ecc71', 'AMST+ViT+HOG': '#e74c3c'}
markers_inv = {'HOG': 'o', 'AMST': 's', 'ViT-B/16': 'D', 'AMST+ViT+HOG': '^'}
titles = {'rotation': 'Rotation (radians)', 'scale': 'Scale Shrink Factor', 'deform': 'Elastic Jitter Magnitude'}
for ax, (transform_type, levels) in zip(axes, [('rotation', rotation_levels), ('scale', scale_levels), ('deform', deform_levels)]):
    for method, per_level in invariance_results[transform_type].items():
        arr = np.array(per_level)
        means = arr.mean(axis=1); stds = arr.std(axis=1)
        ax.errorbar(levels, means, yerr=stds, marker=markers_inv[method], lw=2.2,
                    color=colors_inv[method], markersize=7, capsize=3, label=method)
    ax.set_xlabel(titles[transform_type], fontsize=12, fontweight='bold')
    ax.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
    ax.set_title(f'{transform_type.capitalize()} Invariance', fontsize=13, fontweight='bold')
    ax.set_ylim(0, 100)
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=10, loc='lower left')
plt.suptitle('Controlled Invariance Diagnostic (SGSB)', fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('fig11_invariance_diagnostic.png', dpi=200, bbox_inches='tight')
plt.show()
print('Figure 11 saved.')


Invariance: deform: 100%|██████████| 6/6 [09:54<00:00, 99.10s/it]


Figure 11 saved.


## Cell 27 — Noise Robustness Evaluation

Robustness to image-space noise is evaluated for all four descriptors (HOG, AMST, ViT-B/16, and AMST+ViT+HOG) under an identical corruption protocol. For each noise level, a random subset of test images is corrupted with salt-and-pepper noise followed by Gaussian smoothing, and each descriptor is fully regenerated from the corrupted image before classification with its corresponding pre-trained SVM. Each noise level is repeated across five independent random subsets to report mean accuracy with standard-deviation error bars.


In [33]:
N_ROBUST_TEST = 60          # images sampled per noise level (was 30)
N_ROBUST_REPEATS = 5        # independent repeats per level (was 1 -> no error bars before)
noise_levels = np.linspace(0, 0.3, 7)
print(f'Noise levels: {noise_levels}')

def apply_image_noise(bw, noise_level, rng):
    noisy = bw.copy().astype(np.float32)
    mask = rng.random(bw.shape) < noise_level
    noisy[mask] = rng.choice([0, 1], size=mask.sum())
    if noise_level > 0:
        sigma = noise_level * 2
        noisy = ndimage.gaussian_filter(noisy, sigma)
        noisy = (noisy > 0.5).astype(np.float32)
    return noisy

# --- fit clean-data classifiers exactly as used for the headline results ---
scaler_h = StandardScaler()
X_hog_clean = scaler_h.fit_transform(np.nan_to_num(X_hog))
clf_h = SVC(kernel='rbf', C=100, gamma='scale', decision_function_shape='ovr', random_state=SEED)
clf_h.fit(X_hog_clean, y_mpeg)

COMP_DIMS_LIST = [160, 90, 210, 128, 30]
X_amst_n = np.zeros_like(X_amst)
start = 0
for dim in COMP_DIMS_LIST:
    end = start + dim
    mu = X_amst[:, start:end].mean(axis=0); std = X_amst[:, start:end].std(axis=0) + 1e-10
    X_amst_n[:, start:end] = (X_amst[:, start:end] - mu) / std
    start = end
sel_a = SelectKBest(f_classif, k=350)
X_amst_sel = sel_a.fit_transform(X_amst_n, y_mpeg)
scaler_a = StandardScaler()
X_amst_clean = scaler_a.fit_transform(X_amst_sel)
clf_a = SVC(kernel='rbf', C=100, gamma='scale', decision_function_shape='ovr', random_state=SEED)
clf_a.fit(X_amst_clean, y_mpeg)

scaler_v = StandardScaler()
X_vit_clean_full = scaler_v.fit_transform(np.nan_to_num(X_vit))
clf_v = SVC(kernel='rbf', C=100, gamma='scale', decision_function_shape='ovr', random_state=SEED)
clf_v.fit(X_vit_clean_full, y_mpeg)

sel_c = SelectKBest(f_classif, k=1000)
X_comb_n_full = np.zeros_like(X_comb_mpeg)
start = 0
for dim in COMBINED_COMP_DIMS:
    end = start + dim
    mu = X_comb_mpeg[:, start:end].mean(axis=0); std = X_comb_mpeg[:, start:end].std(axis=0) + 1e-10
    X_comb_n_full[:, start:end] = (X_comb_mpeg[:, start:end] - mu) / std
    start = end
X_comb_sel_full = sel_c.fit_transform(X_comb_n_full, y_mpeg)
scaler_c = StandardScaler()
X_comb_clean_full = scaler_c.fit_transform(X_comb_sel_full)
clf_c = SVC(kernel='rbf', C=100, gamma='scale', decision_function_shape='ovr', random_state=SEED)
clf_c.fit(X_comb_clean_full, y_mpeg)

# --- reload ViT backbone so corrupted images can be re-embedded (kept alive for Cell 28 too) ---
_vit_robust = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0).to(DEVICE).eval()
_tfms_robust = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
def _vit_feat(bw):
    rgb = np.stack([bw]*3, axis=-1).astype(np.float32)
    img = transforms.ToPILImage()(rgb)
    x = _tfms_robust(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        return _vit_robust(x).cpu().numpy().flatten()

noise_results = {'HOG': [], 'AMST': [], 'ViT-B/16': [], 'AMST+ViT+HOG': []}

for noise_lvl in tqdm(noise_levels, desc='Noise evaluation (unified protocol)'):
    level_scores = {k: [] for k in noise_results}
    for rep in range(N_ROBUST_REPEATS):
        rng = np.random.RandomState(SEED * 1000 + rep)
        test_idx = rng.choice(len(all_images), min(N_ROBUST_TEST, len(all_images)), replace=False)
        hog_f, amst_f, vit_f = [], [], []
        for idx in test_idx:
            bw_noisy = apply_image_noise(all_images[idx], noise_lvl, rng)
            c_noisy = get_contour(bw_noisy)
            hog_f.append(hog_descriptor(bw_noisy))
            amst_f.append(amst_descriptor(bw_noisy, c_noisy))
            vit_f.append(_vit_feat(bw_noisy))
        hog_f = np.nan_to_num(np.array(hog_f))
        amst_f = np.nan_to_num(np.array(amst_f))
        vit_f = np.nan_to_num(np.array(vit_f))
        comb_f = np.concatenate([amst_f, vit_f, hog_f], axis=1)
        y_test = y_mpeg[test_idx]

        level_scores['HOG'].append(clf_h.score(scaler_h.transform(hog_f), y_test) * 100)

        amst_n = np.zeros_like(amst_f); start = 0
        for dim in COMP_DIMS_LIST:
            end = start + dim
            mu = amst_f[:, start:end].mean(axis=0); std = amst_f[:, start:end].std(axis=0) + 1e-10
            amst_n[:, start:end] = (amst_f[:, start:end] - mu) / std
            start = end
        level_scores['AMST'].append(clf_a.score(scaler_a.transform(sel_a.transform(amst_n)), y_test) * 100)

        level_scores['ViT-B/16'].append(clf_v.score(scaler_v.transform(vit_f), y_test) * 100)

        comb_n = np.zeros_like(comb_f); start = 0
        for dim in COMBINED_COMP_DIMS:
            end = start + dim
            mu = comb_f[:, start:end].mean(axis=0); std = comb_f[:, start:end].std(axis=0) + 1e-10
            comb_n[:, start:end] = (comb_f[:, start:end] - mu) / std
            start = end
        level_scores['AMST+ViT+HOG'].append(clf_c.score(scaler_c.transform(sel_c.transform(comb_n)), y_test) * 100)

    for k in noise_results:
        noise_results[k].append(level_scores[k])

# --- persist raw results so Fig 6 and Fig 9 both draw from the SAME numbers ---
noise_records = []
for method, per_level in noise_results.items():
    for lvl, scores in zip(noise_levels, per_level):
        for s in scores:
            noise_records.append({'Method': method, 'NoiseLevel': float(lvl), 'Accuracy': s})
pd.DataFrame(noise_records).to_csv('noise_robustness_raw.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 7))
noise_markers = ['o', 's', 'D', '^']
noise_colors_n = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c']
for idx, method in enumerate(noise_results):
    arr = np.array(noise_results[method])   # shape (n_levels, N_ROBUST_REPEATS)
    means = arr.mean(axis=1)
    stds = arr.std(axis=1)
    ax.errorbar(noise_levels*100, means, yerr=stds, marker=noise_markers[idx], lw=2.5,
                color=noise_colors_n[idx], markersize=8, capsize=4, label=method)
    ax.fill_between(noise_levels*100, means-stds, means+stds, alpha=0.08, color=noise_colors_n[idx])
ax.set_xlabel('Noise Level (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=14, fontweight='bold')
ax.set_title(f'Noise Robustness (n={N_ROBUST_TEST} images, {N_ROBUST_REPEATS} repeats/level, all methods unified)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='lower left', frameon=True, fancybox=True)
ax.grid(alpha=0.3)
ax.set_ylim(0, 100)
sns.despine()
plt.tight_layout()
plt.savefig('fig6_noise_robustness.png', dpi=200, bbox_inches='tight')
plt.show()
print('Figure 6 saved.')


Noise levels: [0.   0.05 0.1  0.15 0.2  0.25 0.3 ]


Noise evaluation (unified protocol): 100%|██████████| 7/7 [22:10<00:00, 190.10s/it]


Figure 6 saved.


## Cell 28 — Occlusion Robustness Evaluation

Robustness to spatial occlusion is evaluated using the same protocol as Cell 27: a random square patch covering a target fraction of the image is blacked out, each descriptor is regenerated from the occluded image, and results are averaged over five independent repeats per occlusion level for all four methods.


In [34]:
occ_levels = np.linspace(0, 0.3, 7)
print(f'Occlusion levels: {occ_levels}')

def apply_occlusion(bw, occ_level, rng):
    occ = bw.copy()
    h, w = occ.shape
    patch = max(1, int(np.sqrt(occ_level) * min(h, w)))
    x = rng.randint(0, w - patch + 1)
    y = rng.randint(0, h - patch + 1)
    occ[y:y+patch, x:x+patch] = 0
    return occ

occ_results = {'HOG': [], 'AMST': [], 'ViT-B/16': [], 'AMST+ViT+HOG': []}

for occ_lvl in tqdm(occ_levels, desc='Occlusion evaluation (unified protocol)'):
    level_scores = {k: [] for k in occ_results}
    for rep in range(N_ROBUST_REPEATS):
        rng = np.random.RandomState(SEED * 2000 + rep)
        test_idx = rng.choice(len(all_images), min(N_ROBUST_TEST, len(all_images)), replace=False)
        hog_f, amst_f, vit_f = [], [], []
        for idx in test_idx:
            bw_occ = apply_occlusion(all_images[idx], occ_lvl, rng)
            c_occ = get_contour(bw_occ)
            hog_f.append(hog_descriptor(bw_occ))
            amst_f.append(amst_descriptor(bw_occ, c_occ))
            vit_f.append(_vit_feat(bw_occ))
        hog_f = np.nan_to_num(np.array(hog_f))
        amst_f = np.nan_to_num(np.array(amst_f))
        vit_f = np.nan_to_num(np.array(vit_f))
        comb_f = np.concatenate([amst_f, vit_f, hog_f], axis=1)
        y_test = y_mpeg[test_idx]

        level_scores['HOG'].append(clf_h.score(scaler_h.transform(hog_f), y_test) * 100)

        amst_n = np.zeros_like(amst_f); start = 0
        for dim in COMP_DIMS_LIST:
            end = start + dim
            mu = amst_f[:, start:end].mean(axis=0); std = amst_f[:, start:end].std(axis=0) + 1e-10
            amst_n[:, start:end] = (amst_f[:, start:end] - mu) / std
            start = end
        level_scores['AMST'].append(clf_a.score(scaler_a.transform(sel_a.transform(amst_n)), y_test) * 100)

        level_scores['ViT-B/16'].append(clf_v.score(scaler_v.transform(vit_f), y_test) * 100)

        comb_n = np.zeros_like(comb_f); start = 0
        for dim in COMBINED_COMP_DIMS:
            end = start + dim
            mu = comb_f[:, start:end].mean(axis=0); std = comb_f[:, start:end].std(axis=0) + 1e-10
            comb_n[:, start:end] = (comb_f[:, start:end] - mu) / std
            start = end
        level_scores['AMST+ViT+HOG'].append(clf_c.score(scaler_c.transform(sel_c.transform(comb_n)), y_test) * 100)

    for k in occ_results:
        occ_results[k].append(level_scores[k])

# free the reloaded ViT backbone now that both robustness cells are done
del _vit_robust
if torch.cuda.is_available():
    torch.cuda.empty_cache()

occ_records = []
for method, per_level in occ_results.items():
    for lvl, scores in zip(occ_levels, per_level):
        for s in scores:
            occ_records.append({'Method': method, 'OcclusionLevel': float(lvl), 'Accuracy': s})
pd.DataFrame(occ_records).to_csv('occlusion_robustness_raw.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 7))
occ_markers = ['o', 's', 'D', '^']
occ_colors = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c']
for idx, method in enumerate(occ_results):
    arr = np.array(occ_results[method])
    means = arr.mean(axis=1)
    stds = arr.std(axis=1)
    ax.errorbar(occ_levels*100, means, yerr=stds, marker=occ_markers[idx], lw=2.5,
                color=occ_colors[idx], markersize=8, capsize=4, label=method)
    ax.fill_between(occ_levels*100, means-stds, means+stds, alpha=0.08, color=occ_colors[idx])
ax.set_xlabel('Occlusion Level (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=14, fontweight='bold')
ax.set_title(f'Occlusion Robustness (n={N_ROBUST_TEST} images, {N_ROBUST_REPEATS} repeats/level, all methods unified)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='lower left', frameon=True, fancybox=True)
ax.grid(alpha=0.3)
ax.set_ylim(0, 100)
sns.despine()
plt.tight_layout()
plt.savefig('fig7_occlusion_robustness.png', dpi=200, bbox_inches='tight')
plt.show()
print('Figure 7 saved.')


Occlusion levels: [0.   0.05 0.1  0.15 0.2  0.25 0.3 ]


Occlusion evaluation (unified protocol): 100%|██████████| 7/7 [21:51<00:00, 187.36s/it]


Figure 7 saved.


## Figure 8 — Feature Extraction Time & Dimensionality


In [35]:
complexity_data = {}
for name, X_data, n_samples in [('HOG', X_hog, 1), ('Zernike', X_zern, 1),
                                  ('Fourier', X_four, 1), ('Wavelet', X_wav, 1),
                                  ('CSS', X_css, 1), ('SC', X_sc, 1),
                                   ('AMST', X_amst, 1)]:
    times = []
    for _ in range(5):
        t0 = time.time()
        if name == 'HOG': hog_descriptor(all_images[0])
        elif name == 'Zernike': zernike_descriptor(all_images[0])
        elif name == 'Fourier': fourier_descriptor(all_contours[0])
        elif name == 'Wavelet': wavelet_descriptor(all_contours[0])
        elif name == 'CSS': css_descriptor(all_contours[0])
        elif name == 'SC': shape_context(all_contours[0])
        elif name == 'AMST': amst_descriptor(all_images[0], all_contours[0])
        times.append((time.time()-t0)*1000)
    complexity_data[name] = {'time_ms': np.mean(times), 'std': np.std(times), 'dim': X_data.shape[1], 'feature_name': name}

for name, model_fn in [('ViT-B/16', 'vit_base_patch16_224'), ('ResNet50', 'resnet50'), ('EfficientNet', 'efficientnet_b0')]:
    times = []
    for _ in range(3):
        m = timm.create_model(model_fn, pretrained=False, num_classes=0).eval()
        t0 = time.time()
        x = img_to_tensor(all_images[0]).to(DEVICE)
        m = m.to(DEVICE)
        with torch.no_grad(): m(x)
        times.append((time.time()-t0)*1000)
        del m
    complexity_data[name] = {'time_ms': np.mean(times), 'std': np.std(times), 'dim': 0, 'feature_name': name}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
sns.set_style('whitegrid')
names_c = list(complexity_data.keys())
times_c = [complexity_data[n]['time_ms'] for n in names_c]
dims_c = [complexity_data[n]['dim'] for n in names_c]
colors_c = ['#e74c3c' if 'AMST' in n else ('#2ecc71' if n in ('ViT-B/16','ResNet50','EfficientNet') else '#3498db') for n in names_c]
bars1 = ax1.bar(names_c, times_c, color=colors_c, edgecolor='black', linewidth=1.1, width=0.6)
for bar, v in zip(bars1, times_c):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(times_c)*0.01, f'{v:.1f}',
            ha='center', va='bottom', fontsize=8)
ax1.set_ylabel('Extraction Time (ms)', fontsize=14, fontweight='bold')
ax1.set_title('Per-Image Feature Extraction Time', fontsize=14, fontweight='bold')
ax1.tick_params(axis='x', rotation=45, labelsize=10)
ax1.grid(axis='y', alpha=0.3)
bars2 = ax2.bar(names_c, dims_c, color=colors_c, edgecolor='black', linewidth=1.1, width=0.6)
for bar, v in zip(bars2, dims_c):
    if v > 0:
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(dims_c)*0.005, f'{v}',
                ha='center', va='bottom', fontsize=8)
ax2.set_ylabel('Feature Dimensionality', fontsize=14, fontweight='bold')
ax2.set_title('Feature Vector Size', fontsize=14, fontweight='bold')
ax2.tick_params(axis='x', rotation=45, labelsize=10)
ax2.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig8_complexity_analysis.png', dpi=200, bbox_inches='tight')
plt.show()
print('Figure 8 saved.')


Figure 8 saved.


## Table 1 — Comparison with Prior Work on MPEG-7 CE-Shape-1

MPEG-7 CE-Shape-1 has historically been evaluated under two distinct protocols that are not numerically interchangeable: the **Bullseye score** (the top-40 retrieval hit-rate used by classical shape-matching work such as Shape Context and IDSC) and **supervised classification accuracy** (the protocol used elsewhere in this paper). Table 1a compares against the retrieval literature on the Bullseye metric, using this paper's own Bullseye scores computed in Cell 24. Table 1b reports classification accuracy under this paper's own 10-fold protocol; no external classification-accuracy baselines are included, as no directly comparable published classification protocol (same class split and CV scheme) on this dataset was identified.

**Table 1a — Retrieval protocol (Bullseye score, top-40 hits)**

| Method | Bullseye Score | Reference |
|---|---|---|
| Shape Context | 76.51% | Belongie & Malik, ICCV 2001 / IEEE TPAMI 2002 |
| IDSC + Dynamic Programming | 85.40% | Ling & Jacobs, IEEE TPAMI 2007 |
| HOG (ours) | 56.82% | This paper, Cell 24 |
| ViT-B/16 (ours) | 66.65% | This paper, Cell 24 |
| AMST (ours) | 56.03% | This paper, Cell 24 |
| **AMST+ViT+HOG (ours)** | **68.76%** | This paper, Cell 24 |

Under the standard Bullseye protocol, none of the descriptors evaluated in this paper — including the proposed AMST+ViT+HOG combination — exceed the two classical baselines shown above. This is reported directly rather than omitted: it indicates that the strong MAP and classification-accuracy results elsewhere in this paper do not, by themselves, establish a new state of the art on the *retrieval* task as conventionally measured on this benchmark. The combined descriptor's relative advantage over the individual AMST/ViT/HOG components still holds under the Bullseye metric (68.76% vs. the next-best 66.65%), which is the comparison this paper is positioned to make.

**Table 1b — Classification protocol (supervised accuracy, this paper's own protocol)**

| Method | Accuracy | Protocol |
|---|---|---|
| HOG (SVM-RBF) | 89.79% ± 2.73% | 10-fold stratified CV |
| AMST (SVM-RBF, per-component normalization + Fisher selection) | 92.14% ± 1.89% | 10-fold stratified CV |
| ViT-B/16 (SVM-RBF) | 94.29% ± 1.94% | 10-fold stratified CV |
| **AMST+ViT+HOG (SVM-RBF, per-component normalization + Fisher selection)** | **96.36% ± 1.58%** | 10-fold stratified CV |

**Beyond MPEG-7:** Cells 29-31 report classification accuracy on an independent, fully synthetic shape benchmark (SGSB) together with a controlled diagnostic that varies rotation, scale, and elastic deformation independently. On SGSB, all four methods score highly (HOG 83.00%, AMST 98.00%, ViT-B/16 100.00% +/- 0.00%, AMST+ViT+HOG 99.50%); ViT-B/16 reaching a perfect, zero-variance score indicates that SGSB's ten well-separated polygon classes are near ceiling for a strong pretrained backbone and the benchmark should be read as a sanity check that generalization is not catastrophically broken on a new shape distribution, not as a discriminative comparison between methods (see Limitations). The controlled invariance diagnostic (Figure 11) is more informative: AMST and AMST+ViT+HOG degrade far less than HOG under scale change (e.g., at 30% shrinkage, AMST 93.3% and Combined 100.0% versus HOG's 40.0%) and under elastic deformation (at magnitude 0.096, AMST 50.7% and Combined 89.3% versus HOG's 36.7%), supporting the intended scale- and deformation-robustness of the AMST design. The rotation condition instead produces a non-monotonic curve for every method, including HOG, because several SGSB classes (square, hexagon, octagon, the two star shapes, cross, ellipse) have rotational symmetry, so accuracy partially recovers at rotation angles that are multiples of a class's symmetry period rather than decaying smoothly with angle; this is a property of the specific polygon set used, not evidence against rotation invariance, and should be described as such rather than presented as a clean monotonic robustness curve.


## Figure 9 — Multi-Dimensional Radar Chart

The four descriptors are compared jointly across six axes: classification accuracy, retrieval MAP, cross-dataset MAP, noise robustness at 10% corruption, occlusion robustness at 10% coverage, and feature compactness. Noise and occlusion values are drawn directly from the unified image-space evaluation in Cells 27-28, so all four methods are compared under an identical corruption protocol. Compactness is defined as 100 x (1 - d / d_max), where d is the effective (post-selection) feature dimensionality used by the classifier for that method and d_max is the largest such dimensionality among the four methods.


In [36]:
NOISE_10PCT_IDX = 2   # noise_levels[2] == 0.10 -> 10% noise
OCC_10PCT_IDX = 2     # occ_levels[2]   == 0.10 -> 10% occlusion

radar_methods = ['HOG', 'ViT-B/16', 'AMST', 'AMST+ViT+HOG']
radar_categories = ['Accuracy', 'MAP', 'Cross-MAP', 'Noise@10%', 'Occ@10%', 'Compactness']

# Effective (post-selection) dimensionality actually consumed by the classifier for each method.
# Compactness = 100 * (1 - effective_dim / max_effective_dim); higher = more compact representation.
effective_dim = {
    'HOG': X_hog.shape[1],          # 324-d, no feature selection applied
    'ViT-B/16': X_vit.shape[1],     # 768-d, no feature selection applied
    'AMST': 350,                     # SelectKBest k=350 (Cell 17/18 protocol)
    'AMST+ViT+HOG': 1000,            # SelectKBest k=1000 (Cell 17 protocol)
}
max_dim_for_scale = max(effective_dim.values())

radar_values = []
for m in radar_methods:
    acc = results.get(m, {}).get('mean', 0)
    ret = retrieval_results.get(m, 0) * 100
    cross = cross_results.get(m, 0) * 100 if cross_results else 0
    noise_at_10 = float(np.array(noise_results[m][NOISE_10PCT_IDX]).mean())
    occ_at_10 = float(np.array(occ_results[m][OCC_10PCT_IDX]).mean())
    compact = 100 * (1 - effective_dim[m] / max_dim_for_scale)
    radar_values.append([acc, ret, cross, noise_at_10, occ_at_10, compact])

angles = np.linspace(0, 2*np.pi, len(radar_categories), endpoint=False).tolist()
angles += angles[:1]
radar_colors_fig = {'HOG': '#3498db', 'ViT-B/16': '#2ecc71', 'AMST': '#e67e22', 'AMST+ViT+HOG': '#e74c3c'}
radar_markers_fig = {'HOG': 'o', 'ViT-B/16': 's', 'AMST': 'D', 'AMST+ViT+HOG': '^'}
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
for method in radar_methods:
    vals = radar_values[radar_methods.index(method)]
    vals_plot = vals + vals[:1]
    ax.plot(angles, vals_plot, 'o-', lw=2.5, color=radar_colors_fig[method],
            marker=radar_markers_fig[method], markersize=8, label=method, alpha=0.9)
    ax.fill(angles, vals_plot, alpha=0.08, color=radar_colors_fig[method])
ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_categories, fontsize=12, fontweight='bold')
ax.set_ylim(0, 100)
ax.set_yticks([20, 40, 60, 80, 100])
ax.set_yticklabels(['20', '40', '60', '80', '100'], fontsize=9)
ax.set_title('Multi-Dimensional Comparison of Shape Descriptors\n'
             '(Noise/Occ: unified image-space protocol; Compactness = 100*(1 - dim/max_dim))',
             fontsize=13, fontweight='bold', pad=30)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=11, frameon=True, fancybox=True)
plt.tight_layout()
plt.savefig('fig9_radar_chart.png', dpi=200, bbox_inches='tight')
plt.show()
print('Figure 9 saved.')


Figure 9 saved.


## Cell 32 — Results Summary


In [37]:
print('='*70)
print('EXPERIMENTAL RESULTS SUMMARY')
print('='*70)

best_method_name = max(results, key=lambda n: results[n]['mean'])
best_acc = results[best_method_name]['mean']
best_std = results[best_method_name]['std']
best_map_method = max(retrieval_results, key=lambda n: retrieval_results.get(n, 0))
best_cross_method = max(cross_results, key=lambda n: cross_results.get(n, 0)) if cross_results else ''

print(f'\n{"="*60}')
print('EXPERIMENTAL RESULTS SUMMARY')
print(f'{"="*60}')
print(f'\n--- Classification (10-fold CV, SVM-RBF) ---')
for nm in sorted(results, key=lambda n: results[n]['mean'], reverse=True):
    marker = ' *' if nm == best_method_name else '  '
    print(f'{marker} {nm:30s}: {results[nm]["mean"]:6.2f}% +/-{results[nm]["std"]:4.2f}%')
print(f'  Friedman p={friedman_p:.4f} ({"significant" if friedman_p<0.05 else "not significant"})')

print(f'\n--- Shape Retrieval (MAP, cosine similarity) ---')
for nm in sorted(retrieval_results, key=lambda n: retrieval_results[n], reverse=True):
    marker = ' *' if nm == best_map_method else '  '
    print(f'{marker} {nm:30s}: {retrieval_results[nm]*100:6.2f}%')

print(f'\n--- Cross-Dataset Generalization (MAP, MPEG-7 -> Kimia-216) ---')
for nm in sorted(cross_results, key=lambda n: cross_results[n], reverse=True):
    marker = ' *' if nm == best_cross_method else '  '
    print(f'{marker} {nm:30s}: {cross_results[nm]*100:6.2f}%')

results_df = pd.DataFrame([{'Method': nm, 'Accuracy': f'{r["mean"]:.2f}%', 'Std': f'{r["std"]:.2f}%'}
                            for nm, r in results.items()])
results_df.to_csv('classification_results.csv', index=False)
print('\nResults saved to classification_results.csv')

ret_df = pd.DataFrame([{'Method': nm, 'MAP': f'{v*100:.2f}%'} for nm, v in retrieval_results.items()])
ret_df.to_csv('retrieval_results.csv', index=False)
print('Retrieval results saved to retrieval_results.csv')

if cross_results:
    cross_df = pd.DataFrame([{'Method': nm, 'Cross_MAP': f'{v*100:.2f}%'} for nm, v in cross_results.items()])
    cross_df.to_csv('cross_dataset_results.csv', index=False)
    print('Cross-dataset results saved to cross_dataset_results.csv')

print(f'\n{"="*70}')
print(f'Figures generated: fig1 through fig9 (9 total)')
print(f'{"="*70}')


EXPERIMENTAL RESULTS SUMMARY

EXPERIMENTAL RESULTS SUMMARY

--- Classification (10-fold CV, SVM-RBF) ---
 * AMST+ViT+HOG                  :  96.36% +/-1.58%
   ViT-B/16                      :  94.29% +/-1.94%
   AMST                          :  92.14% +/-1.89%
   HOG                           :  89.79% +/-2.73%
  Friedman p=0.0001 (significant)

--- Shape Retrieval (MAP, cosine similarity) ---
 * AMST+ViT+HOG                  :  67.29%
   ViT-B/16                      :  63.63%
   HOG                           :  54.95%
   AMST                          :  51.65%

--- Cross-Dataset Generalization (MAP, MPEG-7 -> Kimia-216) ---
 * HOG                           :  34.50%
   AMST+ViT+HOG                  :  16.71%
   ViT-B/16                      :  16.07%
   AMST                          :   6.43%

Results saved to classification_results.csv
Retrieval results saved to retrieval_results.csv
Cross-dataset results saved to cross_dataset_results.csv

Figures generated: fig1 through fig9 (9 to

In [38]:
import shutil
import os

output_dir = 'results_output'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

figure_files = [
    'fig1_mpeg7_dataset.png',
    'fig2_accuracy_comparison.png',
    'fig3_confusion_matrix.png',
    'fig4_precision_recall.png',
    'fig5_cross_dataset.png',
    'fig6_noise_robustness.png',
    'fig7_occlusion_robustness.png',
    'fig8_complexity_analysis.png',
    'fig9_radar_chart.png',
    'fig10_sgsb_samples.png',
    'fig11_invariance_diagnostic.png',
]
for f in figure_files:
    if os.path.exists(f):
        shutil.move(f, os.path.join(output_dir, f))

csv_files = [
    'classification_results.csv',
    'retrieval_results.csv',
    'cross_dataset_results.csv',
    'bullseye_results.csv',
    'noise_robustness_raw.csv',
    'occlusion_robustness_raw.csv',
    'hyperparameter_grid_search.csv',
    'cross_dataset_ablation.csv',
    'cross_dataset_per_class_amst.csv',
    'sgsb_classification_results.csv',
    'invariance_diagnostic_raw.csv',
]
for f in csv_files:
    if os.path.exists(f):
        shutil.move(f, os.path.join(output_dir, f))

zip_filename = 'all_results'
shutil.make_archive(zip_filename, 'zip', output_dir)
print(f"All results saved to {zip_filename}.zip")


All results saved to all_results.zip


## Discussion

### Key Findings

The proposed **AMST+ViT+HOG** hybrid descriptor achieves the strongest results on the MPEG-7 CE-Shape-1 benchmark under this paper's evaluation protocols:

1. **Classification Accuracy (96.36%)**: The combined descriptor outperforms individual components (ViT-B/16: 94.29%, AMST: 92.14%, HOG: 89.79%). The Friedman test indicates a statistically significant difference across methods (p=0.0001), with Nemenyi post-hoc analysis identifying which pairs differ significantly; paired t-tests are reported as a supplementary, descriptive statistic. The shared SVM hyperparameters (C=100, gamma='scale') are validated by the grid search in Cell 19, which found no held-out accuracy improvement from a searched configuration over the fixed default.
2. **Shape Retrieval (67.29% MAP; 68.76% Bullseye score, Cell 24)**: The global cosine similarity on the concatenated feature space preserves the discriminative power of ViT while benefiting from complementary AMST and HOG features, achieving a 3.66 percentage point MAP improvement and a 2.11 percentage point Bullseye improvement over ViT alone. Under the standard Bullseye protocol, this result remains below the classical Shape Context (76.51%) and IDSC+DP (85.40%) baselines (Table 1a); the contribution of this paper is therefore framed as improving on deep and handcrafted single-descriptor baselines under a consistent evaluation protocol, not as establishing a new retrieval state of the art relative to the classical shape-matching literature.
3. **Cross-Dataset Generalization**: HOG's gradient features generalize best across domains (34.50% MAP on MPEG-7 -> Kimia-216), consistent with their domain-agnostic, low-level nature. The combined descriptor (16.71%) outperforms both ViT (16.07%) and AMST alone (6.43%). The component-wise diagnostic (Cell 26) shows that cross-dataset MAP falls monotonically as more AMST components are added: C1 alone reaches 11.44% Cross-MAP, C1+C3 drops to 9.53%, adding C2 (the config labelled "C1-C3: +Topology") drops further to 8.22%, and adding C4 drops to 5.97%, before the full five-component descriptor recovers slightly to 6.43%. This is the opposite of the within-dataset ablation, where C3 (SPD Riemannian) contributed the single largest gain (+20.36pp, see Component Contribution below): the components that most improve within-dataset accuracy (C3, C4) are also the components most responsible for the cross-dataset generalization loss, consistent with those higher-order components fitting MPEG-7-specific shape statistics rather than transferable structure. The per-class breakdown (`cross_dataset_per_class_amst.csv`) shows this failure is highly uneven rather than uniform: 16 of the 18 shared classes have a cross-dataset AP below 5% (several below 2%, e.g., hammer at 0.96%, fork at 1.07%), while two classes transfer much better (face at 25.09%, glas at 50.65%), indicating that AMST's cross-dataset weakness is concentrated in specific shape categories rather than affecting all classes equally.
4. **Generalization Beyond MPEG-7**: The SGSB classification results (Cell 30) indicate whether the relative ordering of methods observed on MPEG-7 (Combined > ViT > AMST > HOG) is specific to that benchmark's shape distribution or holds more broadly.

### Component Contribution

The ablation study indicates that each AMST component contributes to the final performance:
- **C1 (APCFW+)**: 65.43% — a strong radial-frequency baseline
- **+C3 (SPD Riemannian)**: +20.36pp — covariance structure adds substantial discriminative power
- **+C2 (Topology)**: +2.14pp — persistence features provide complementary information
- **+C4 (Morphology)**: +3.79pp — morphological profiles capture structural detail
- **+C5 (Complexity)**: +0.43pp — high-level shape properties provide a small further gain

### Comparison with Deep Learning

ViT-B/16 alone achieves 94.29%, demonstrating the strength of transformer-based visual features on this task. The proposed hybrid approach improves on this by 2.07pp, indicating that handcrafted shape descriptors (AMST, HOG) encode complementary geometric information not fully captured by deep features alone.


## Limitations and Future Work

### Current Limitations

1. **Cross-Dataset Generalization**: AMST alone drops from 51.65% MAP within MPEG-7 to 6.43% MAP on the MPEG-7 -> Kimia-216 transfer task. The component-wise diagnostic (Cell 26) traces this to the higher-order components (C3 SPD Riemannian, C4 Morphology): cross-dataset MAP falls monotonically from 11.44% (C1 alone) to 5.97% as these components are added, indicating that the components most responsible for within-dataset accuracy gains are also the ones least transferable across datasets. The per-class breakdown further shows that this failure is concentrated in 16 of 18 shared classes (AP below 5%), while two classes (face, glas) transfer comparatively well, so the appropriate remedy is class- and component-targeted (e.g., regularizing or down-weighting C3/C4 for transfer-sensitive classes) rather than a general robustness fix.
2. **Benchmark Scope**: The primary results are established on MPEG-7 CE-Shape-1 (1,400 images, 70 classes), with Kimia-216 (216 images, 18 classes) as a cross-dataset probe and SGSB (200 images, 10 classes, procedurally generated) as a controlled, dataset-independent check. SGSB classification accuracy is uniformly high across all four methods (HOG 83.00%, AMST 98.00%, ViT-B/16 100.00% +/- 0.00%, AMST+ViT+HOG 99.50%), which means SGSB demonstrates the absence of catastrophic failure on a new shape distribution but is at ceiling and does not discriminate meaningfully between methods; it should not be cited as evidence that the combined descriptor outperforms ViT-B/16 in general, since ViT-B/16 alone already saturates the benchmark. The controlled invariance diagnostic (Figure 11) is the more informative part of this section: it shows a genuine scale- and deformation-robustness advantage for AMST and the combined descriptor over HOG, while the rotation condition is confounded by the rotational symmetry of several SGSB polygon classes and should be interpreted with that caveat rather than as a clean monotonic curve. SGSB and Kimia-216 together are a complement to, not a substitute for, evaluation on additional real-world shape benchmarks (e.g., Swedish Leaf, Flavia, ETH-80, or the Princeton Shape Benchmark), which remains valuable future work.
3. **Protocol Comparability**: classification accuracy, MAP, and Bullseye score are not interchangeable (Table 1). Under the standard Bullseye protocol, this paper's best result (68.76%, AMST+ViT+HOG) remains below the classical Shape Context (76.51%) and IDSC+DP (85.40%) baselines; this paper's contribution should be described as improving over its own deep and handcrafted single-descriptor baselines under a consistent protocol, not as a new retrieval state of the art relative to the classical shape-matching literature.
4. **Hyperparameter Sensitivity**: the grid search (Cell 19) found an identical 3-fold CV accuracy (93.93%) for every tested C in {10, 50, 100, 200, 500} with either gamma setting, and a lower accuracy (92.23%) only at C=1; this indicates the fixed default (C=100, gamma='scale') used throughout the paper sits within a broad, stable optimum rather than being an arbitrarily chosen value, though it also means this particular grid search cannot distinguish among C in {10, ..., 500} and a finer-grained or differently regularized search would be needed to rule out overfitting at the high end of that range.
5. **Computational Complexity**: AMST feature extraction takes approximately 10 minutes for 1,400 images on CPU, compared to under 2 seconds for HOG. The noise, occlusion, and controlled invariance evaluations each re-embed several hundred to a few thousand corrupted images through ViT-B/16 and are the most time-consuming cells in this notebook on CPU-only hardware.
6. **Feature Dimensionality**: The combined 1,710-dimensional descriptor requires feature selection (SelectKBest, k=1000) prior to classification, adding preprocessing overhead.

### Future Research Directions

1. **Domain Adaptation**: Incorporating domain adaptation techniques (e.g., CORAL, MMD), targeted specifically at components C3 and C4, could improve cross-dataset generalization by aligning feature distributions across domains without discarding the within-dataset accuracy gains those components provide.
2. **A Harder Synthetic Benchmark**: A revised SGSB with visually confusable classes and larger deformation ranges would avoid the ceiling effect observed here and provide a more discriminative synthetic complement to MPEG-7.
3. **Lightweight AMST Variant**: A reduced-dimensionality AMST variant (targeting 300-400 dimensions) could improve computational efficiency while retaining discriminative power.
4. **End-to-End Learning**: Learnable, differentiable versions of the AMST components could enable end-to-end fine-tuning for specific downstream tasks.
5. **Additional Real-World Benchmarks**: Extending validation to further real-world shape datasets beyond MPEG-7 and Kimia-216 would complement the SGSB results and further strengthen the generality of the reported findings.


## Conclusion

This paper presents **AMST+ViT+HOG**, a hybrid shape descriptor that combines Adaptive Multi-Scale Shape Topology (AMST) features with deep learning (ViT-B/16) and classical gradient features (HOG) for shape classification and retrieval. Through per-component normalization and Fisher feature selection, the method achieves **96.36% classification accuracy** under a 10-fold stratified cross-validation protocol on MPEG-7 CE-Shape-1 and **67.29% retrieval MAP (68.76% Bullseye score)**. Under the standard Bullseye protocol these retrieval results remain below the classical Shape Context and IDSC+DP baselines (Table 1a); the paper's retrieval contribution is therefore the consistent improvement of the combined descriptor over its individual deep and handcrafted components, evaluated under a single protocol, rather than a new state of the art relative to the classical literature.

The key contributions are:
1. A five-component AMST descriptor capturing radial-frequency, topological, Riemannian, morphological, and complexity-based shape properties, with each component's contribution quantified via ablation.
2. A fusion strategy combining handcrafted and deep features with per-component normalization and Fisher feature selection, validated with a hyperparameter sensitivity analysis (Cell 19).
3. Classification and retrieval results on MPEG-7 CE-Shape-1 with statistical significance verification via the Friedman and Nemenyi tests, and a component-wise, factor-wise diagnosis of the method's cross-dataset generalization gap (Cells 25-26, 31).
4. A unified noise/occlusion robustness evaluation across all four methods (Cells 27-28), and an independent classification and controlled-invariance check on a procedurally generated synthetic benchmark (Cells 29-31).

The source code, preprocessed features, and full experimental results are made available for reproducibility.


## Reproducibility Statement

All experiments were conducted with a fixed random seed (SEED=42) using 10-fold stratified cross-validation (5-fold for the smaller SGSB benchmark). The complete source code, preprocessed features, and experimental results — classification, retrieval, Bullseye, cross-dataset (aggregate, component-wise, and per-class), ablation, hyperparameter sensitivity, noise/occlusion robustness, and SGSB classification/invariance results — are provided alongside this notebook. All computation is CPU-compatible: PyTorch and timm automatically fall back to CPU execution when no CUDA device is available, and no cell requires GPU-specific operations. Key libraries: PyTorch 2.9.0, timm 1.0.27, scikit-learn 1.6.1, NumPy 2.0.2. Expected total runtime on a CPU-only environment is summarized at the top of the notebook.
